![1_1778427665610.png](./1_1778427665610.png "1_1778427665610.png")

![2_1778427655020.png](./2_1778427655020.png "2_1778427655020.png")

 
 ![3_1778427630609.png](./3_1778427630609.png "3_1778427630609.png")

![4_1778427727409.png](./4_1778427727409.png "4_1778427727409.png")

# Introduction to Spark Structured Streaming — Lecture Notes (Professor Style)

Welcome back.

Before we jump deep into Spark Structured Streaming APIs, I want to make sure you understand *why stream processing exists* and *why it is difficult*.

Because once you understand the problems, Spark Structured Streaming will suddenly make a lot more sense.

You already know Spark programming.

You already know Spark DataFrames.

You already know Spark SQL.

Now the question is:

> How do we use all that knowledge in the world of continuous, real-time processing?

That is exactly what this lecture is about.

In this lecture, we will focus on two major things:

1. **When do we need stream processing?**
2. **What new challenges appear when we move from batch processing to stream processing?**

And trust me — the second point is the most important.

Because stream processing is not just “batch processing running faster.”

It introduces entirely new engineering problems.

---

# Understanding Stream Processing with a Real Example

Let’s take a real-world scenario.

Assume you are working for a large stock brokerage company.

This company provides multiple platforms for customers to trade stocks:

* Desktop trading terminals
* Web applications
* Laptop applications
* Mobile apps

Customers can buy and sell stocks using any of these systems.

Every trade generates transaction data.

Now the business comes to your data engineering team with a requirement.

---

# Business Requirement

The company wants a **live trade summary dashboard**.

The dashboard should show:

* Total Buy Amount
* Total Sell Amount
* Settlement Difference

over time.

---

# Understanding the Dashboard

Imagine a graph.

## X-Axis → Time

* 9:00
* 9:15
* 9:30
* 9:45
* 10:00
* and so on...

## Y-Axis → Amount

* 5000
* 10000
* 15000
* 20000
* etc.

Now:

* Blue Line → Total Buy Amount
* Orange Line → Total Sell Amount
* Gray Line → Settlement Difference

At any given point in time:

* Buy line shows total stock purchases
* Sell line shows total stock sales
* Difference line shows settlement amount

That’s the reporting requirement.

Simple enough.

But now the real question begins.

---

# Thinking Like a Data Engineer

As data engineers, we usually divide every solution into 3 stages.

---

# Stage 1 — Data Ingestion

We first ask:

> How do we collect the data?

The trading systems are generating transactions continuously.

So somehow we need to ingest that data into our data platform.

---

# Stage 2 — Data Processing

Once the data is available inside our platform:

* Clean it
* Transform it
* Aggregate it
* Enrich it
* Prepare final results

---

# Stage 3 — Data Consumption

Finally:

* Store results
* Expose them to dashboards
* Allow BI tools to consume them

This is the standard data engineering lifecycle.

---

# Designing the Batch Processing Solution

Now let’s design the solution step by step.

---

# Step 1 — Assume a Central Collection System

Let’s simplify things.

Assume there is already a “magical system” collecting trades from:

* mobile apps
* desktop terminals
* web applications
* laptops

We are NOT building that system.

We simply assume it already exists.

So all trade data is already available centrally.

Good.

---

# Step 2 — Build a Landing Zone

Now we create a **Landing Zone**.

A landing zone is simply a directory where raw files arrive.

We build an ingestion job.

The job:

1. Connects to the central system
2. Pulls trade data
3. Saves it as files

Maybe:

* every 15 minutes
* every 5 minutes
* every hour

depending on the requirement.

So our first pipeline becomes:

```text
Trading Systems
      ↓
Central System
      ↓
Landing Zone (Raw Files)
```

---

# Step 3 — Bronze Layer (Raw Table)

Now we read data from the landing zone.

And save it into a Bronze table.

This becomes our **source of truth**.

```text
Landing Zone
      ↓
Bronze Table
```

The Bronze layer contains raw data exactly as received.

---

# Step 4 — Silver Layer (Cleaned Data)

Next:

We clean and standardize the data.

Examples:

* Remove invalid records
* Fix formatting
* Handle missing values
* Standardize timestamps

Then save it into a Silver table.

```text
Bronze
      ↓
Silver
```

Silver = high-quality trusted data.

---

# Step 5 — Gold Layer (Business Results)

Finally:

We aggregate and transform the Silver data into business-ready results.

For example:

* total buy amount
* total sell amount
* settlement values

Then save those results into Gold tables.

```text
Silver
      ↓
Gold
```

This Gold layer powers dashboards and reports.

---

# Complete Medallion Architecture

So the final architecture becomes:

```text
Trading Apps
      ↓
Central System
      ↓
Landing Zone
      ↓
Bronze Layer
      ↓
Silver Layer
      ↓
Gold Layer
      ↓
Dashboard
```

This is the classic Medallion Architecture.

---

# Pipeline Orchestration

Now comes another challenge.

How do we run all these jobs?

We need orchestration.

Maybe using:

* Airflow
* Databricks Workflows
* Azure Data Factory
* Luigi
* Oozie

and so on.

So our pipeline becomes:

```text
Job 0 → Ingestion
Job 1 → Bronze
Job 2 → Silver
Job 3 → Aggregation
Job 4 → Gold
```

All jobs run in sequence.

---

# The Most Important Question

Now comes the most important question.

## What is the pipeline frequency?

Should the pipeline run:

* Monthly?
* Daily?
* Hourly?
* Every 15 minutes?
* Every second?

THIS determines whether batch processing is enough.

Or whether we need stream processing.

---

# When Batch Processing Works Well

If your frequency is:

* Daily
* Weekly
* Monthly
* Hourly

then life is easy.

Why?

Because you have enough time between iterations.

Example:

```text
Run pipeline
↓
Finish processing
↓
Wait 1 hour
↓
Run again
```

Simple.

No pressure.

---

# When Problems Start Appearing

Now imagine the requirement changes.

The customer says:

> Refresh dashboard every 15 minutes.

Suddenly things become harder.

Why?

Because now:

* Cluster startup time matters
* Processing speed matters
* Data volume matters
* Cleanup activities matter

Everything must finish within 15 minutes.

---

# Example of the Timing Problem

Suppose:

* Cluster startup = 5 minutes
* Processing = 8 minutes
* Cleanup = 3 minutes

Total:

```text
5 + 8 + 3 = 16 minutes
```

But requirement is 15 minutes.

You already failed.

---

# Understanding Back Pressure

Now imagine this keeps happening repeatedly.

Each iteration becomes delayed.

Example:

```text
Iteration 1 → delayed by 2 min
Iteration 2 → delayed by 4 min
Iteration 3 → delayed by 7 min
```

The delay keeps growing.

This is called:

# Back Pressure

Back pressure means:

> Data arrives faster than your system can process it.

This is one of the biggest problems in stream processing.

---

# Problem 2 — Continuous Scheduling

Now imagine requirements become even more aggressive.

Instead of 15 minutes:

* every second
* every millisecond
* ASAP

Now traditional workflow scheduling becomes impossible.

How do you continuously listen for new data?

How do you trigger jobs instantly?

This is another major challenge.

---

# Problem 3 — Incremental Processing

To improve performance:

we cannot process everything repeatedly.

We must process only NEW data.

This is called:

# Incremental Processing

Instead of:

```text
Process entire dataset every time
```

we do:

```text
Process only newly arrived records
```

This dramatically improves speed.

---

# But Incremental Processing Needs Memory

Now the system must remember:

* what was already processed
* what is new

This introduces another concept:

# Checkpointing

Checkpointing stores progress information.

---

# Understanding Checkpointing

Example:

Iteration 1 processes:

```text
File A
File B
```

Checkpoint stores:

```text
Processed = A, B
```

Next iteration:

New files arrive:

```text
A, B, C, D
```

Using checkpoint:

System knows:

```text
A and B already processed
Only process C and D
```

That is incremental processing.

---

# Problem 4 — Fault Tolerance

Now imagine iteration 11 fails.

Iterations 1–10 succeeded.

What should happen after restart?

Correct behavior:

```text
Restart from iteration 11
```

NOT from beginning.

NOT from iteration 10.

This requires reliable checkpointing.

---

# The Transaction Problem

Now things become tricky.

You actually have TWO transactions happening:

1. Data Processing
2. Checkpoint Writing

What if:

* data processing succeeds
* but checkpoint write fails?

Or opposite?

Now your system becomes inconsistent.

So stream processing systems must ensure:

```text
Either BOTH succeed
OR BOTH fail
```

This is a very complex engineering problem.

Spark Structured Streaming handles this internally.

---

# Problem 5 — Late Arriving Data

This is one of the hardest concepts.

Let’s understand carefully.

Suppose:

At 10:00 AM you calculate:

```text
Total Sales = 3400
```

based on all transactions between:

```text
9:30 → 10:00
```

Good.

Now at 10:30 AM:

a delayed transaction suddenly arrives.

But its timestamp is:

```text
9:59 AM
```

This transaction belonged to the PREVIOUS window.

Your earlier calculation is now wrong.

---

# Why Late Data Is Dangerous

At 10:00 AM:

your result was correct based on available data.

But later:

new information arrived.

Now previous result becomes invalid.

So you must:

1. Recalculate previous window
2. Correct old results
3. Update aggregates

This is called:

# State Management

The system must remember previous calculations and update them when late data arrives.

Very difficult problem.

---

# Summary — The 5 Major Challenges of Stream Processing

As frequency becomes smaller:

```text
Hours → Minutes → Seconds → Milliseconds
```

these problems become critical.

---

# 1. Back Pressure

System cannot process data fast enough.

---

# 2. Continuous Scheduling

Need always-on processing.

---

# 3. Incremental Processing

Process only new records.

Requires checkpointing.

---

# 4. Fault Tolerance

Need reliable restart and recovery.

---

# 5. Late Arriving Data + State Management

Need to correct previously calculated results.

---

# Final Understanding

The goal of this lecture is NOT to scare you.

The goal is to help you understand:

> Stream processing is fundamentally different from batch processing.

When frequencies become very small:

* complexity increases dramatically
* engineering challenges increase dramatically

Now imagine if a framework could solve all these problems for you automatically.

That is exactly why Spark Structured Streaming exists.

It provides built-in solutions for:

* checkpointing
* fault tolerance
* incremental processing
* scheduling
* back pressure handling
* late data handling
* state management

And throughout this course, we will learn how to use those capabilities properly.

---

# Key Takeaways

## Batch Processing

Good for:

* daily jobs
* weekly jobs
* monthly jobs

---

## Stream Processing

Needed when:

* low latency matters
* near real-time analytics required
* continuous ingestion required

---

## Main Challenges

* Back pressure
* Scheduling
* Checkpointing
* Fault tolerance
* Late-arriving data
* State management

---

# Mental Model to Remember Forever

Think of Batch Processing like:

```text
Take snapshot → Process → Stop
```

Think of Stream Processing like:

```text
Continuously listen → Continuously process → Continuously update
```

That is the core difference.

---

# End of Lecture

That’s all for this lecture.

In the next lecture, we will start understanding how Spark Structured Streaming actually solves these problems internally using Structured Streaming APIs.

Keep learning and keep growing.


In [0]:
%python
'''Databricks notebook source '''
#01-streaming-word-count.py

class batchWC():
    def __init__(self):
        self.base_data_dir = "/FileStore/data_spark_streaming_scholarnest"

    def getRawData(self):
        from pyspark.sql.functions import explode, split
        lines = (spark.read
                    .format("text")
                    .option("lineSep", ".")
                    .load(f"{self.base_data_dir}/data/text")
                )
        return lines.select(explode(split(lines.value, " ")).alias("word"))
    
    def getQualityData(self, rawDF):
        from pyspark.sql.functions import trim, lower
        return ( rawDF.select(lower(trim(rawDF.word)).alias("word"))
                        .where("word is not null")
                        .where("word rlike '[a-z]'")
                )
        
    def getWordCount(self, qualityDF):
        return qualityDF.groupBy("word").count()
    
    def overwriteWordCount(self, wordCountDF):
        ( wordCountDF.write
                    .format("delta")
                    .mode("overwrite")
                    .saveAsTable("word_count_table")
        )
    
    def wordCount(self):
        print(f"\tExecuting Word Count...", end='')
        rawDF = self.getRawData()
        qualityDF = self.getQualityData(rawDF)
        resultDF = self.getWordCount(qualityDF)
        self.overwriteWordCount(resultDF)
        print("Done")

# COMMAND ----------

class streamWC():
    def __init__(self):
        self.base_data_dir = "/FileStore/data_spark_streaming_scholarnest"

    def getRawData(self):
        from pyspark.sql.functions import explode, split
        lines = (spark.readStream
                    .format("text")
                    .option("lineSep", ".")
                    .load(f"{self.base_data_dir}/data/text")
                )
        return lines.select(explode(split(lines.value, " ")).alias("word"))
    
    def getQualityData(self, rawDF):
        from pyspark.sql.functions import trim, lower
        return ( rawDF.select(lower(trim(rawDF.word)).alias("word"))
                        .where("word is not null")
                        .where("word rlike '[a-z]'")
                )
        
    def getWordCount(self, qualityDF):
        return qualityDF.groupBy("word").count()
    
    def overwriteWordCount(self, wordCountDF):
        return ( wordCountDF.writeStream
                    .format("delta")
                    .option("checkpointLocation", f"{self.base_data_dir}/chekpoint/word_count")
                    .outputMode("complete")
                    .toTable("word_count_table")
                )
    
    def wordCount(self):
        print(f"\tStarting Word Count Stream...", end='')
        rawDF = self.getRawData()
        qualityDF = self.getQualityData(rawDF)
        resultDF = self.getWordCount(qualityDF)
        sQuery = self.overwriteWordCount(resultDF)
        print("Done")
        return sQuery

# COMMAND ----------

# MAGIC %md
# MAGIC &copy; 2021-2023 <a href="https://www.scholarnest.com/">ScholarNest Technologies Pvt. Ltd. </a>All rights reserved.<br/>
# MAGIC <br/>
# MAGIC <a href="https://www.scholarnest.com/privacy/">Privacy Policy</a> | <a href="https://www.scholarnest.com/terms/">Terms of Use</a> | <a href="https://www.scholarnest.com/contact-us/">Contact Us</a>

In [0]:
%python
# 2
# Databricks notebook source
# MAGIC %run ./01-streaming-word-count

# COMMAND ----------

class batchWCTestSuite():
    def __init__(self):
        self.base_data_dir = "/FileStore/data_spark_streaming_scholarnest"

    def cleanTests(self):
        print(f"Starting Cleanup...", end='')
        spark.sql("drop table if exists word_count_table")
        dbutils.fs.rm("/user/hive/warehouse/word_count_table", True)

        dbutils.fs.rm(f"{self.base_data_dir}/chekpoint", True)
        dbutils.fs.rm(f"{self.base_data_dir}/data/text", True)

        dbutils.fs.mkdirs(f"{self.base_data_dir}/data/text")
        print("Done\n")

    def ingestData(self, itr):
        print(f"\tStarting Ingestion...", end='')
        dbutils.fs.cp(f"{self.base_data_dir}/datasets/text/text_data_{itr}.txt", f"{self.base_data_dir}/data/text/")
        print("Done")

    def assertResult(self, expected_count):
        print(f"\tStarting validation...", end='')
        actual_count = spark.sql("select sum(count) from word_count_table where substr(word, 1, 1) == 's'").collect()[0][0]
        assert expected_count == actual_count, f"Test failed! actual count is {actual_count}"
        print("Done")

    def runTests(self):
        self.cleanTests()
        wc = batchWC()

        print("Testing first iteration of batch word count...") 
        self.ingestData(1)
        wc.wordCount()
        self.assertResult(25)
        print("First iteration of batch word count completed.\n")

        print("Testing second iteration of batch word count...") 
        self.ingestData(2)
        wc.wordCount()
        self.assertResult(32)
        print("Second iteration of batch word count completed.\n") 

        print("Testing third iteration of batch word count...") 
        self.ingestData(3)
        wc.wordCount()
        self.assertResult(37)
        print("Third iteration of batch word count completed.\n")
    

# COMMAND ----------

bwcTS = batchWCTestSuite()
bwcTS.runTests()

# COMMAND ----------

class streamWCTestSuite():
    def __init__(self):
        self.base_data_dir = "/FileStore/data_spark_streaming_scholarnest"

    def cleanTests(self):
        print(f"Starting Cleanup...", end='')
        spark.sql("drop table if exists word_count_table")
        dbutils.fs.rm("/user/hive/warehouse/word_count_table", True)

        dbutils.fs.rm(f"{self.base_data_dir}/chekpoint", True)
        dbutils.fs.rm(f"{self.base_data_dir}/data/text", True)

        dbutils.fs.mkdirs(f"{self.base_data_dir}/data/text")
        print("Done\n")

    def ingestData(self, itr):
        print(f"\tStarting Ingestion...", end='')
        dbutils.fs.cp(f"{self.base_data_dir}/datasets/text/text_data_{itr}.txt", f"{self.base_data_dir}/data/text/")
        print("Done")

    def assertResult(self, expected_count):
        print(f"\tStarting validation...", end='')
        actual_count = spark.sql("select sum(count) from word_count_table where substr(word, 1, 1) == 's'").collect()[0][0]
        assert expected_count == actual_count, f"Test failed! actual count is {actual_count}"
        print("Done")

    def runTests(self):
        import time
        sleepTime = 30

        self.cleanTests()
        wc = streamWC()
        sQuery = wc.wordCount()

        print("Testing first iteration of batch word count...") 
        self.ingestData(1)
        print(f"\tWaiting for {sleepTime} seconds...") 
        time.sleep(sleepTime)
        self.assertResult(25)
        print("First iteration of batch word count completed.\n")

        print("Testing second iteration of batch word count...") 
        self.ingestData(2)
        print(f"\tWaiting for {sleepTime} seconds...") 
        time.sleep(sleepTime)
        self.assertResult(32)
        print("Second iteration of batch word count completed.\n") 

        print("Testing third iteration of batch word count...") 
        self.ingestData(3)
        print(f"\tWaiting for {sleepTime} seconds...") 
        time.sleep(sleepTime)
        self.assertResult(37)
        print("Third iteration of batch word count completed.\n")

        sQuery.stop()
    

# COMMAND ----------

swcTS = streamWCTestSuite()
swcTS.runTests()

# COMMAND ----------

# MAGIC %md
# MAGIC &copy; 2021-2023 <a href="https://www.scholarnest.com/">ScholarNest Technologies Pvt. Ltd. </a>All rights reserved.<br/>
# MAGIC <br/>
# MAGIC <a href="https://www.scholarnest.com/privacy/">Privacy Policy</a> | <a href="https://www.scholarnest.com/terms/">Terms of Use</a> | <a href="https://www.scholarnest.com/contact-us/">Contact Us</a>

# Building Your First Streaming Word Count Application — Professor Style Notes

---

# Introduction — What Are We Building?

Welcome back.

In this lecture, we are starting something very important — our **first streaming solution**.

But instead of directly jumping into streaming, we will move step by step like real data engineers.

The overall learning journey of this lecture is:

1. Understand the problem statement.
2. Build a batch processing solution first.
3. Identify the problems in batch processing.
4. Understand why streaming is needed.
5. Later, transform the batch solution into a streaming solution.

This is exactly how real-world systems evolve.

Nobody starts with streaming immediately.

First, people build batch pipelines.
Then business requirements become faster.
Then streaming architecture is introduced.

So this lecture is laying the foundation for stream processing.

---

# Problem Statement — Word Count Application

We want to create a simple **Word Count Application**.

The idea is straightforward:

* Some text files will arrive in a landing zone directory.
* Our Spark application will read those files.
* We will process the data.
* Then calculate word counts.
* Finally save the result into a table.

---

# Architecture Overview

The overall flow looks like this:

```text
Landing Zone (data/text)
        ↓
Read Files
        ↓
Create Raw DataFrame
        ↓
Apply Data Quality Rules
        ↓
Calculate Word Count
        ↓
Save Into Word Count Table
        ↓
Consumer Reporting
```

---

# Understanding the Landing Zone

We create a directory:

```text
data/text
```

This directory acts as the **landing zone**.

What is a landing zone?

A landing zone is the location where source systems drop incoming data files.

Our application begins processing from this directory.

---

# What Happens in First Iteration?

In the first iteration:

1. Read all files from landing zone.
2. Create raw DataFrame.
3. Apply data quality checks.
4. Perform word count aggregation.
5. Save result into target table.

Simple.

---

# Understanding the Final Report

The final report is extremely simple.

* X-axis → Alphabet letters
* Y-axis → Count of words

Example:

| Alphabet | Count |
| -------- | ----- |
| A        | 120   |
| S        | 250   |
| M        | 90    |

Meaning:

* How many words start with A
* How many start with S
* And so on

---

# Iterative Processing Concept

After first execution:

* New files will arrive again.
* We process only new files.
* Recalculate word count.
* Update target table.

This repeated execution is called **iterative batch processing**.

---

# Initial Setup in Databricks

Before writing code, we need two things:

## 1. Compute Cluster

We create a Databricks compute cluster.

Without cluster:

* Spark code cannot execute.

---

## 2. Sample Data

We upload sample files into DBFS.

Path:

```text
DBFS → FileStore
```

We upload:

```text
sample_data/
```

This directory contains text files used for testing.

---

# Creating Workspace and Notebook

Inside Databricks Workspace:

1. Go to Home Directory
2. Create new folder
3. Create notebook

This notebook will contain our Spark application.

---

# Step 1 — Read Data From Landing Zone

First task:

Read files from:

```text
base_directory/data/text
```

Code logic:

* File format = text
* Each line becomes one record

Spark loads text data into a DataFrame column called:

```python
value
```

Important:

Every line from file becomes one row.

---

# Creating Raw DataFrame

Now we start processing.

The logic:

```text
Line → Split into words → Convert array into rows
```

---

# Step 1 — Split Words

Suppose line is:

```text
Spark is amazing
```

We split by spaces.

Result:

```python
["Spark", "is", "amazing"]
```

This creates an array.

---

# Step 2 — Explode Array

Arrays are not useful for aggregation.

We need rows.

So we use:

```python
explode()
```

Result:

| word    |
| ------- |
| Spark   |
| is      |
| amazing |

Now each word becomes an individual row.

This becomes our **raw words DataFrame**.

---

# Data Quality Rules

Raw data may contain garbage.

Examples:

* Null values
* Numbers
* Special symbols
* Spaces
* Mixed case text

So we create a **quality DataFrame**.

---

# Quality Rule 1 — Trim Spaces

We remove unwanted spaces using:

```python
trim()
```

Example:

```text
" Spark "
```

becomes:

```text
"Spark"
```

---

# Quality Rule 2 — Convert to Lowercase

We standardize all words using:

```python
lower()
```

Example:

```text
SPARK
Spark
spark
```

All become:

```text
spark
```

This improves aggregation accuracy.

---

# Quality Rule 3 — Remove Nulls

We eliminate empty records.

Example:

```text
""
null
```

These should not participate in counting.

---

# Quality Rule 4 — Regex Filtering

We only keep words that start with alphabets.

Examples removed:

```text
123hello
#spark
@data
```

Only valid alphabetic words survive.

---

# Aggregation Step — Word Count

Once quality data is ready:

We perform aggregation.

Logic:

```python
groupBy(word).count()
```

Example:

| word  | count |
| ----- | ----- |
| spark | 50    |
| data  | 30    |

This becomes our final result DataFrame.

---

# Writing Results Into Target Table

Now we save output.

We use:

```python
.write
.format("delta")
.mode("overwrite")
.saveAsTable("word_count_table")
```

---

# Why Delta Format?

Delta format provides:

* Reliability
* ACID transactions
* Better performance
* Versioning
* Streaming compatibility

Very important for production systems.

---

# Initial Setup Notebook

Before running application:

We must clean environment.

Why?

Because previous runs may leave:

* old tables
* old checkpoints
* old files

These can corrupt testing.

---

# Cleanup Activities

We perform:

## Drop Existing Table

```sql
DROP TABLE IF EXISTS word_count_table
```

---

## Delete Directories

We delete:

```text
wordcount table directory
checkpoint directory
data/text directory
```

---

## Recreate Landing Zone

Finally:

```text
data/text
```

is recreated.

Now environment is clean.

---

# Running the Application

After setup:

We execute notebook step by step.

All stages execute successfully:

1. Read data
2. Process words
3. Apply quality rules
4. Aggregate counts
5. Save table

Pipeline works.

---

# Important Software Engineering Discussion

Now comes a VERY important point.

The instructor says:

> Writing sequential notebook code is NOT a recommended production approach.

This is critical.

---

# Problem With Straight Notebook Coding

If everything is written line-by-line:

```python
read data
process
aggregate
write
```

Then:

* Hard to test
* Hard to debug
* Hard to reuse
* Hard to automate

Production systems should be modular.

---

# Solution — Refactor Into Functions

We restructure code into:

* Class
* Methods
* Small reusable functions

This makes code:

* Testable
* Maintainable
* Reusable
* Production ready

---

# BatchWordCount Class

We create class:

```python
BatchWordCount
```

Constructor stores:

```python
base_data_dir
```

---

# Function 1 — getRawData()

Purpose:

* Read files
* Split words
* Return raw words DataFrame

This function only does ONE responsibility.

Excellent software design practice.

---

# Function 2 — getQualityData()

Purpose:

* Apply quality checks
* Remove invalid data
* Return clean DataFrame

Again:

Single responsibility.

---

# Function 3 — getWordCount()

Purpose:

* Aggregate words
* Return count DataFrame

---

# Function 4 — overwriteWordCount()

Purpose:

* Save final results into target table

---

# Main Orchestration Method

We create method:

```python
wordCount()
```

This stitches all functions together.

Execution flow:

```text
getRawData()
      ↓
getQualityData()
      ↓
getWordCount()
      ↓
overwriteWordCount()
```

This is orchestration.

---

# Why This Design Is Powerful

Because now every method can be tested independently.

This is the foundation of:

* Unit testing
* Integration testing
* CI/CD pipelines
* Production engineering

---

# Writing Automated Test Cases

Now we start creating a test suite.

This is a very professional engineering practice.

---

# Why Automated Testing?

Manual testing is unreliable.

Automated tests provide:

* Repeatability
* Speed
* Reliability
* Confidence

---

# Creating Test Suite Class

We create:

```python
BatchWordCountTestSuite
```

Inside this class:

* setup methods
* cleanup methods
* ingestion methods
* validation methods

---

# cleanTests() Method

Purpose:

* Remove old environment
* Reset system

This ensures:

Every test starts fresh.

Very important.

---

# ingestData(iteration)

This method copies test files into landing zone.

Iteration parameter controls:

| Iteration | File            |
| --------- | --------------- |
| 1         | text_data_1.txt |
| 2         | text_data_2.txt |
| 3         | text_data_3.txt |

This simulates incoming batches.

---

# assertResult(expectedCount)

This validates correctness.

Logic:

* Query target table
* Calculate actual count
* Compare with expected count

If mismatch:

Test fails.

---

# Why Assertions Matter

Assertions verify:

> “Did system produce expected result?”

Without assertions:

You are not testing.
You are only executing code.

Huge difference.

---

# runTests() — Main Test Execution

This method executes everything.

Flow:

```text
Clean Environment
      ↓
Create WordCount Object
      ↓
Ingest Data
      ↓
Run Processing
      ↓
Validate Results
```

---

# Importing Notebook Using %run

To access class definitions:

```python
%run ./streaming_word_count
```

This imports notebook code into current notebook.

Very common Databricks technique.

---

# First Iteration Testing

Steps:

1. Ingest first file
2. Run batch processing
3. Validate expected count

Expected:

```python
25 words starting with 's'
```

Assertion passes.

---

# Second Iteration Testing

Repeat process with second file.

Expected count:

```python
32
```

Passes again.

---

# Third Iteration Testing

Repeat again.

Expected:

```python
37
```

Passes successfully.

---

# Important Engineering Principle

Instructor mentions:

> Minimum two iterations should be tested.

Why?

Because batch systems are incremental.

We must verify:

* Initial execution works
* Subsequent executions work

This validates incremental logic.

---

# Final Batch Architecture

At this point:

We have:

✅ Batch application
✅ Proper structure
✅ Refactored code
✅ Automated tests
✅ Iterative execution
✅ Production-style engineering

Excellent foundation.

---

# Scheduling the Workflow

Now comes orchestration.

We create:

## Job 1

Ingest data from source systems into landing zone.

---

## Job 2

Run word count processing.

---

# Pipeline Scheduling

Pipeline is scheduled every:

```text
15 minutes
```

This becomes a complete batch workflow.

---

# BIG Problem With Batch Processing

Now comes the key architectural discussion.

What if business wants:

* updates every minute?
* every second?
* immediately?

Batch processing starts becoming inefficient.

---

# Challenges of Reducing Batch Frequency

Suppose we reduce frequency from:

```text
15 min → 1 min → 1 sec
```

Problems appear:

## 1. High Resource Consumption

Repeated job startup cost.

---

## 2. Increased Latency

Still not real-time.

---

## 3. Expensive Scheduling

Too many executions.

---

## 4. Reprocessing Complexity

Handling incremental files becomes difficult.

---

# Need for Streaming

Business now wants:

> “As soon as data arrives, refresh report immediately.”

This requirement leads to:

## Stream Processing

Streaming systems continuously monitor incoming data and process it instantly.

---

# Core Transition

This lecture teaches a very important industry evolution:

```text
Batch Processing
        ↓
Near Real Time
        ↓
Streaming Architecture
```

This is how modern data platforms evolve.

---

# Key Learning Summary

## Technical Concepts Learned

* Landing zones
* Raw DataFrames
* Data quality processing
* Word count aggregation
* Delta tables
* Refactoring
* Class-based design
* Unit testing
* Incremental processing
* Workflow scheduling

---

# Engineering Principles Learned

## Good Code Must Be:

* Modular
* Testable
* Reusable
* Structured
* Maintainable

---

# Most Important Takeaway

The most important lesson from this lecture is:

> Streaming is not just a technology upgrade.
> It is a response to business latency requirements.

Batch works fine until business demands faster insights.

That demand creates streaming systems.

---

# Final Mental Model

Think of the system like this:

```text
Files Arrive
      ↓
Batch Job Reads Files
      ↓
Processes Data
      ↓
Produces Report
```

But later:

```text
Data Arrives
      ↓
System Reacts Immediately
      ↓
Streaming Processing Happens
      ↓
Report Updates Instantly
```

That transformation is the journey from:

## Batch → Streaming

And that is exactly where the next lecture begins.

---

# End of Lecture Notes

Next Topic:

## Transforming Batch Word Count Into Streaming Word Count Application


# Transforming Batch Processing into Stream Processing — Professor Style Lecture Notes

---

# Introduction — From Batch to Streaming

Welcome back.

In the previous lecture, we built a **batch processing solution** for the word count problem.

The architecture looked something like this:

```text id="8u5i4z"
Landing Zone
      ↓
Read Data
      ↓
Process Data
      ↓
Calculate Word Count
      ↓
Save to Target Table
```

This is a **typical data engineering workflow**.

Almost every real-world project follows this pattern:

* ingest data
* process data
* save results

Very standard architecture.

---

# Original Business Requirement

Initially, the customer wanted:

```text id="b4wg9f"
Run pipeline every 15 minutes
```

That sounds reasonable.

Most batch systems can easily handle this.

But now comes the real challenge.

---

# New Requirement — Reduce Frequency

Suppose customer says:

```text id="zvd7k0"
“I want fresh results every 30 seconds.”
```

Now think carefully.

Can the same batch architecture sustain this requirement?

Maybe initially yes.

But over time?

Probably not.

And that becomes the core problem of batch systems.

---

# The Hidden Problem in Batch Processing

The instructor identifies a major issue:

## Full Load Processing

This entire architecture performs:

# FULL LOAD + FULL PROCESS + FULL OVERWRITE

---

# Understanding Full Load

Let us understand this deeply.

---

# First Iteration

Suppose landing zone contains:

```text id="9pp0wu"
2 files
```

System reads:

```text id="r4p66s"
ALL 2 FILES
```

Processes them.

Writes results.

---

# Second Iteration

Now 2 more files arrive.

Landing zone now contains:

```text id="o9a92n"
4 files total
```

What does batch job do?

It again reads:

```text id="1k2c9h"
ALL 4 FILES
```

Not just new files.

Everything again.

---

# Tenth Iteration

Now maybe 20 files exist.

Batch job again reads:

```text id="bb0jgi"
ALL 20 FILES
```

Every single time.

---

# Core Batch Problem

So what is happening?

Every execution:

1. Reads full dataset
2. Processes full dataset
3. Rewrites full target table

This is extremely inefficient as data grows.

---

# Why Batch Eventually Fails

Initially system is fast because data volume is small.

But over time:

```text id="jltjot"
More files
→ More processing
→ More execution time
→ Increased delays
```

Eventually:

* 15-minute SLA fails
* 30-second SLA becomes impossible

---

# Important Concept — Back Pressure

Instructor introduces a very important term:

# Back Pressure

What does it mean?

Back pressure happens when:

> New incoming data arrives faster than system can process it.

Example:

* Job takes 20 minutes
* But new execution should happen every 15 minutes

System starts falling behind.

This creates processing backlog.

---

# Why This Happens

Because batch architecture performs:

```text id="jy0k6p"
Full Read
Full Process
Full Overwrite
```

again and again.

This does not scale.

---

# The Real Solution — Incremental Processing

So how do we solve this?

The answer is:

# Incremental Data Processing

---

# Understanding Incremental Processing

Instead of processing everything repeatedly:

We process only NEW data.

---

# Incremental Logic

## First Iteration

Process:

```text id="h7rvd0"
File 1
File 2
```

Save results.

---

## Second Iteration

Only process:

```text id="ux0wtx"
NEW files
```

Do NOT reprocess old files.

---

# Final Target Update

Instead of overwriting full table:

We update incrementally.

This dramatically reduces processing cost.

---

# But Incremental Processing Is Hard

Now comes an important realization.

Implementing incremental processing manually is NOT easy.

Challenges:

* tracking processed files
* identifying new files
* managing state
* handling failures
* maintaining consistency

This becomes complex.

---

# Enter Spark Structured Streaming

This is where:

# Spark Structured Streaming

comes into the picture.

Spark streaming simplifies incremental processing enormously.

---

# Goal of This Lecture

We now transform:

```text id="9u0mr6"
Batch Word Count
        ↓
Streaming Word Count
```

using Structured Streaming.

---

# Important Observation

Instructor makes a powerful point:

> We are NOT rewriting entire application.

Only small changes are required.

This is because of Spark’s:

# Unified API

---

# Unified API — Most Important Spark Concept

Spark provides almost identical APIs for:

* Batch processing
* Stream processing

Meaning:

Most transformation logic remains SAME.

---

# Three Stages of Data Processing

Every pipeline has 3 stages:

```text id="jlwmtr"
Read
Process
Write
```

This is universally true.

---

# Batch vs Stream — Key Difference

The middle processing logic remains SAME.

Difference exists only in:

1. How we READ
2. How we WRITE

This is the biggest conceptual takeaway.

---

# Processing Logic Does NOT Change

All transformations remain identical:

* split
* explode
* trim
* lower
* filter
* groupBy
* count

All same.

Only input/output behavior changes.

---

# First Major Change — Read Stream

In batch we used:

```python id="5s7bnk"
spark.read
```

In streaming we use:

```python id="ukp6e7"
spark.readStream
```

Why?

Because now data is continuously arriving.

We are not reading static data anymore.

We are reading a live stream.

---

# What Does readStream Do?

`readStream` continuously watches source directory.

Whenever new data arrives:

Spark automatically picks it up.

Very powerful.

---

# Important Mental Model

Batch read:

```text id="sv4zpp"
Read once and finish
```

Streaming read:

```text id="jplmrm"
Keep listening forever
```

Huge conceptual difference.

---

# Streaming DataFrame

`readStream` creates:

# Streaming DataFrame

This DataFrame behaves differently internally.

Spark knows:

```text id="j6ubzx"
“This data is continuously arriving.”
```

---

# Second Major Change — Write Stream

Batch write:

```python id="m8j7yv"
.write
```

Streaming write:

```python id="1jquup"
.writeStream
```

Again:

because result is now streaming output.

---

# Why Streaming Output Exists

Input is streaming.

Transformations operate continuously.

Therefore output also becomes streaming.

Makes sense.

---

# Critical Requirement — Checkpointing

Now comes the most important streaming concept.

# Checkpointing

---

# Why Checkpointing Is Mandatory

Streaming systems must remember:

* what was already processed
* what is pending
* offsets
* progress
* metadata

Without memory, incremental processing is impossible.

---

# Checkpoint Location

We specify:

```python id="43l49f"
.option("checkpointLocation", path)
```

This directory stores streaming state.

---

# What Does Spark Store There?

Spark automatically stores:

* processed file information
* metadata
* progress tracking
* recovery state

This enables:

# Fault Tolerance + Incremental Processing

---

# Amazing Thing About Structured Streaming

Instructor emphasizes:

> Spark handles checkpointing automatically.

We only specify location.

Spark manages everything internally.

This is why Structured Streaming is so powerful.

---

# API Naming Differences

There are minor naming changes.

---

# Batch

```python id="8z49o3"
.mode("overwrite")
.saveAsTable()
```

---

# Streaming

```python id="z9m52w"
.outputMode("complete")
.toTable()
```

Meaning remains almost same.

Only API names differ slightly.

---

# Understanding outputMode("complete")

Complete mode means:

Every trigger produces complete output table.

Equivalent to overwrite behavior.

---

# Streaming Query Object

Very important difference:

```python id="2cz4qz"
writeStream
```

returns a:

# StreamingQuery object

Unlike batch writes.

---

# Why StreamingQuery Matters

Streaming jobs run continuously in background.

Spark gives us a handle to control it.

Using StreamingQuery we can:

* stop stream
* monitor stream
* manage execution

---

# Returning StreamingQuery

Instructor returns query object from methods.

Why?

Because later during testing we need:

```python id="71q2qz"
query.stop()
```

Very important for controlling lifecycle.

---

# Main Processing Logic Remains SAME

Look carefully.

Inside wordCount():

Nothing changes.

Still:

```text id="ax7pca"
Read
→ Quality
→ Aggregate
→ Write
```

Only source/sink behavior changed.

---

# Most Important Spark Streaming Principle

If you already know Spark batch processing:

# You already know most of Spark streaming.

Only need to learn:

* stream reading
* stream writing
* checkpointing

This is a huge learning advantage.

---

# Testing Streaming Applications

Now comes testing.

Question:

How do we test streaming jobs?

---

# Batch Testing vs Streaming Testing

Batch testing:

```text id="jlwmk0"
Execute → Finish
```

Streaming testing:

```text id="b5v4nd"
Start once → Keep running
```

Very different behavior.

---

# Reusing Batch Test Suite

Instructor copies existing batch test suite.

Why?

Because most logic remains same.

Again demonstrating unified architecture.

---

# What Changes in Streaming Tests?

Almost nothing.

Three things change:

1. Instantiate StreamingWordCount
2. Start stream only ONCE
3. Wait after ingestion

---

# Important Streaming Behavior

Batch jobs:

```text id="m0ahyl"
Run separately for every iteration
```

Streaming jobs:

```text id="vjlwm7"
Start once and keep listening forever
```

This is the core operational difference.

---

# Starting Stream Once

In batch:

```text id="xy8yya"
Iteration 1 → Run Job
Iteration 2 → Run Job
Iteration 3 → Run Job
```

Streaming:

```text id="n0ptvb"
Start Stream Once
→ Stream keeps waiting
→ Processes new data automatically
```

Beautiful architecture.

---

# Continuous Listening

Streaming query behaves like a listener.

Flow:

```text id="brk6cf"
No Data
→ Wait
→ New File Arrives
→ Process Automatically
→ Wait Again
```

This is event-driven architecture.

---

# Why Sleep Is Required in Tests

After ingesting data:

Processing is asynchronous.

We must give stream time to process.

Hence:

```python id="hy7dbx"
time.sleep(30)
```

This allows:

* pickup
* processing
* aggregation
* writing

to complete.

---

# Streaming Assertions

After waiting:

We run same assertions.

Expected counts:

| Iteration | Expected Count |
| --------- | -------------- |
| 1         | 25             |
| 2         | 32             |
| 3         | 37             |

All pass successfully.

---

# Stopping Streaming Query

Very important cleanup step:

```python id="n7gq1q"
query.stop()
```

Otherwise stream keeps running forever.

---

# Final Streaming Lifecycle

Streaming execution flow:

```text id="0g69ec"
Start Stream
      ↓
Wait for Data
      ↓
Data Arrives
      ↓
Process Automatically
      ↓
Write Results
      ↓
Wait Again
```

This continues forever.

---

# Major Architectural Shift

Batch architecture:

```text id="pjrvlv"
Schedule Driven
```

Streaming architecture:

```text id="r5vhnr"
Event Driven
```

This is a huge conceptual transition.

---

# What We Achieved

We transformed:

```text id="b0b11f"
Batch Pipeline
        ↓
Streaming Pipeline
```

with minimal code changes.

That is the beauty of Spark Structured Streaming.

---

# Key Learning Summary

---

# 1. Batch Processing Problem

Batch repeatedly processes full dataset.

Does not scale well.

---

# 2. Back Pressure

Incoming data eventually exceeds processing capacity.

---

# 3. Incremental Processing

Only process NEW data.

This is the correct scalable approach.

---

# 4. Structured Streaming

Spark simplifies incremental processing using:

* readStream
* writeStream
* checkpointing

---

# 5. Unified API

Transformations remain SAME between:

* batch
* streaming

Only source and sink APIs change.

---

# 6. Checkpointing

Checkpoint stores stream progress and enables fault tolerance.

---

# 7. StreamingQuery

Streaming jobs return query handles for lifecycle management.

---

# 8. Stream Starts Once

Unlike batch jobs, stream processing keeps running continuously.

---

# Final Mental Model

## Batch Processing

```text id="ojc6ui"
Run Job
→ Process Everything
→ Stop
```

---

## Stream Processing

```text id="s6onlv"
Start Stream Once
→ Keep Listening
→ Process Incrementally
→ Never Stop
```

That is the true difference between:

# Batch vs Streaming

---

# End of Lecture Notes

Next Topic:

## Internal Working of Spark Structured Streaming and Streaming Queries


# Understanding Internals of Spark Structured Streaming — Professor Style Lecture Notes

---

# Introduction — What Happens Behind the Scenes?

Welcome back.

In this lecture, we are going deeper into Spark Structured Streaming.

Until now, we learned:

* how to build streaming applications
* how to convert batch into stream processing
* how to use `readStream`
* how to use `writeStream`

But now we want to understand:

# What actually happens internally?

How does Spark Structured Streaming work behind the scenes?

This lecture is extremely important because once you understand the internals:

* streaming becomes easy
* debugging becomes easier
* checkpointing makes sense
* micro-batches make sense
* streaming queries make sense

So let us start slowly and build the complete mental model.

---

# Data Processing Is Always Three Steps

The instructor begins with a very important universal principle.

Every data processing system follows:

```text id="8mt4tm"
Read Data
    ↓
Process Data
    ↓
Write Result
```

This is true for:

* batch processing
* streaming processing
* Spark
* Flink
* databases
* ETL tools

Everything follows this architecture.

---

# Spark Streaming Is No Different

In Spark Structured Streaming:

## Step 1 — Read Stream

We use:

```python id="jlwm75"
spark.readStream
```

---

## Step 2 — Process Data

Apply transformations:

* split
* filter
* aggregate
* groupBy
* joins
* etc.

---

## Step 3 — Write Stream

Finally:

```python id="5qev2r"
writeStream
```

writes result into:

* table
* Kafka
* console
* file system
* Delta table
* streaming sink

---

# But How Does Spark Execute This?

Now comes the important internal discussion.

Spark does not directly execute your code line by line.

Instead:

Spark converts your code into an:

# Execution Plan

---

# Spark SQL Engine

Instructor reminds us:

Spark takes your code and submits it to:

# Spark SQL Engine

---

# What Happens Inside Spark SQL Engine?

Spark SQL engine performs:

1. Logical planning
2. Optimizations
3. Physical planning
4. DAG generation

Finally:

Spark creates the best execution strategy.

---

# Execution Plan / DAG

The final plan is called:

* Execution Plan
* Physical Plan
* DAG (Directed Acyclic Graph)

These are closely related concepts.

---

# Streaming Execution Plan

Now comes the important difference.

This is NOT a normal Spark execution plan.

This is a:

# Streaming Execution Plan

---

# How Spark Knows It Is Streaming

Spark understands streaming because it sees:

```python id="gmjlwm"
readStream
```

and

```python id="tcz9cq"
writeStream
```

These APIs tell Spark:

```text id="4rqxtp"
“This is a streaming application.”
```

---

# High-Level Streaming Execution Flow

The execution plan conceptually looks like:

```text id="o2v9h0"
readStream
      ↓
Transformations
      ↓
Aggregations
      ↓
writeStream
```

This becomes one complete streaming execution plan.

---

# The Most Important Internal Component

Now instructor introduces the heart of Spark Streaming.

# Streaming Query

This is one of the most important concepts in Structured Streaming.

---

# What Is a Streaming Query?

For every streaming execution plan:

Spark creates a:

# Background Thread

called:

# Streaming Query

---

# Responsibility of Streaming Query

Streaming Query is responsible for:

* monitoring data source
* triggering execution
* coordinating processing
* managing checkpoint
* tracking progress
* executing micro-batches

Think of it as:

# The Manager of Your Streaming Application

---

# Streaming Query Lifecycle

Once execution plan is ready:

Spark Driver starts:

# One Streaming Query

for that plan.

---

# First Activity — Initialize Checkpoint

As soon as streaming query starts:

it initializes:

# Checkpoint Location

---

# What Is Checkpoint?

Checkpoint is simply:

# A directory

used for storing streaming metadata.

---

# Important Clarification

Checkpoint does NOT store actual business data.

Instead it stores:

* metadata
* offsets
* processing status
* progress information
* processed file tracking

---

# After Checkpoint Initialization

Streaming Query now starts monitoring:

# Streaming Source

In our example:

```text id="jlwm0l"
Landing Zone Directory
```

---

# What Happens If No Data Exists?

Suppose landing zone is empty.

What does streaming query do?

It waits.

---

# Important Streaming Behavior

Streaming query continuously watches source directory.

Conceptually:

```text id="z0a4p6"
“Do we have new data?”
```

If answer is NO:

```text id="08iblf"
Wait...
```

---

# Very Important Difference from Batch

Batch processing:

```text id="jlwm4d"
Read once and finish
```

Streaming processing:

```text id="jlwmc9"
Keep watching forever
```

This is the core architectural difference.

---

# First File Arrives

Suppose new file arrives:

```text id="jlwmu7"
file1
```

Streaming query immediately detects it.

Why?

Because it is actively monitoring landing zone.

---

# What Streaming Query Does Next

Now streaming query performs 2 critical tasks.

---

# Step 1 — Update Checkpoint

Streaming query writes metadata into checkpoint:

```text id="jlwm9z"
“file1 is ready for processing”
```

Important:

Spark is NOT copying actual file data into checkpoint.

Only metadata is stored.

---

# What Metadata Is Stored?

Things like:

* filename
* processing status
* batch id
* offsets
* timestamps

---

# Step 2 — Trigger Micro-Batch

After checkpoint update:

Streaming Query triggers execution.

This execution is called:

# Micro-Batch

---

# What Is a Micro-Batch?

This is one of the most important concepts.

# Micro-Batch = One Iteration of Streaming Execution

---

# Important Mental Model

Streaming is actually:

# Continuous sequence of tiny batch jobs

This is why it is called:

# Micro-Batch Processing

---

# What Happens Inside Micro-Batch?

Streaming query configures micro-batch input.

In this case:

```text id="5jlwm"
Input = file1
```

---

# Important Point

Even though landing zone may contain many files:

Streaming query explicitly decides:

```text id="zjlwm"
“This batch should process only file1”
```

---

# readStream Execution

Now micro-batch begins.

`readStream` reads ONLY:

```text id="jlwmr0"
file1
```

because streaming query configured it that way.

---

# Then Transformations Execute

After reading:

All processing logic runs:

* split
* explode
* trim
* lower
* aggregation
* groupBy

Same Spark transformations.

---

# writeStream Execution

Finally:

`writeStream`

saves results into sink:

* table
* Delta
* Kafka
* etc.

---

# Monitoring the Batch

Streaming query continuously monitors execution.

It checks:

```text id="jlwmjg"
Did micro-batch complete successfully?
```

---

# Commit Phase

Once execution succeeds:

Streaming query updates checkpoint again.

Now checkpoint says:

```text id="jlwmfh"
file1 processed successfully
batch completed
```

This is called:

# Commit

---

# Important Reason for Commit

Now Spark remembers:

```text id="mjlwm"
file1 is already processed
```

This is the foundation of:

# Incremental Processing

---

# Streaming Never Stops

Critical concept:

After processing completes:

Application does NOT terminate.

Instead:

Streaming query again starts monitoring source.

---

# Infinite Processing Loop

Streaming applications work like:

```text id="jlwmr7"
Watch
→ Detect
→ Process
→ Commit
→ Repeat Forever
```

This is continuous processing architecture.

---

# Second File Arrives

Suppose now:

```text id="njlwmm"
file2
```

arrives.

Streaming query detects it.

---

# Important Observation

Landing zone still contains:

```text id="rjlwm"
file1
file2
```

Spark did NOT delete old file.

---

# Why File1 Is Not Reprocessed

Because checkpoint already remembers:

```text id="qjlwm"
file1 processed successfully
```

Therefore:

Micro-batch 2 processes ONLY:

```text id="9jlwm"
file2
```

This is incremental processing.

---

# Huge Achievement

Without writing custom incremental logic:

Spark automatically processes only NEW data.

This is the biggest power of Structured Streaming.

---

# Third Iteration Example

Suppose now:

```text id="xjlwm"
file3
file4
```

arrive together.

Streaming query detects both.

---

# Checkpoint Update

Checkpoint stores:

```text id="jlwm92"
Micro-batch 3
Input:
- file3
- file4
```

---

# Micro-Batch 3 Execution

Streaming query triggers batch.

`readStream` reads ONLY:

```text id="7jlwm"
file3
file4
```

Not older files.

---

# Processing Continues

Again:

```text id="ajlwmm"
Read
→ Transform
→ Aggregate
→ Write
→ Commit
```

---

# Then Infinite Waiting Again

After commit:

Streaming query returns to monitoring state.

Waiting for more data forever.

---

# Core Streaming Architecture

This entire system operates as:

```text id="ljlwms"
Infinite Loop of Micro-Batches
```

This is the true architecture of Spark Structured Streaming.

---

# Three Most Important Concepts

Instructor summarizes 3 critical concepts.

You MUST understand these deeply.

---

# 1. Streaming Query

## Definition

Background thread managing streaming execution.

---

# Responsibilities

Streaming query:

* monitors source
* triggers batches
* manages checkpoint
* tracks progress
* commits metadata
* keeps infinite loop running

Think of it as:

# Streaming Orchestrator

---

# 2. Micro-Batch

## Definition

One execution iteration of streaming plan.

---

# Important Idea

Your code:

```text id="6jlwm"
readStream → transformations → writeStream
```

executes repeatedly.

Each execution = one micro-batch.

---

# Very Important Mental Model

Streaming applications are NOT magical continuous code.

Internally they are:

# Repeated tiny Spark jobs

executing continuously.

---

# 3. Checkpoint

## Definition

Directory storing streaming metadata and progress.

---

# Checkpoint Helps Spark Answer

Questions like:

* What is already processed?
* What failed?
* What is pending?
* Which batch succeeded?

---

# Why Checkpoint Is Critical

Without checkpoint:

Spark cannot implement:

# Fault-Tolerant Incremental Processing

Checkpoint is the memory of streaming system.

---

# Biggest Takeaway

Instructor concludes with extremely important statement:

# Structured Streaming Is Implicitly Incremental

Meaning:

Spark automatically processes only NEW data.

You do NOT manually write:

* incremental filters
* processed file tracking
* custom metadata management

Spark gives this out of the box.

---

# Final End-to-End Mental Model

Here is the complete architecture.

---

# Step 1 — Start Stream

```text id="3jlwm"
Streaming Query starts
```

---

# Step 2 — Watch Source

```text id="9jlwmu"
Wait for new files
```

---

# Step 3 — Detect New Data

```text id="tjlwmm"
file arrives
```

---

# Step 4 — Update Checkpoint

```text id="6jlwme"
record metadata
```

---

# Step 5 — Trigger Micro-Batch

```text id="9jlwma"
execute processing
```

---

# Step 6 — Process Only New Data

```text id="0jlwmg"
read only unprocessed files
```

---

# Step 7 — Write Results

```text id="cjlwmh"
writeStream saves output
```

---

# Step 8 — Commit Checkpoint

```text id="2jlwmj"
mark batch successful
```

---

# Step 9 — Repeat Forever

```text id="yjlwmk"
go back to waiting state
```

---

# Ultimate Spark Streaming Formula

The entire lecture can be summarized as:

```text id="rjlwml"
Streaming Query
        +
Checkpoint
        +
Infinite Micro-Batches
        =
Spark Structured Streaming
```

---

# Key Technical Concepts Learned

## Spark Internals

* execution plans
* DAG
* SQL engine

---

## Streaming Components

* streaming query
* micro-batch
* checkpoint

---

## Streaming Behavior

* infinite processing loop
* incremental execution
* source monitoring
* automatic state tracking

---

# Most Important Insight

The most important understanding from this lecture is:

> Spark Structured Streaming is fundamentally a continuously repeating sequence of intelligently managed incremental micro-batches.

Once you understand this:

Everything else in Spark Streaming becomes easier.

---

# End of Lecture Notes

Next Topic:

## Deep Dive Into Checkpointing, Fault Tolerance, and Exactly-Once Processing


# Spark Structured Streaming Project — Invoice Processing Application

## Introduction

Welcome back.

In this lecture, we are going to build a real-world Spark Structured Streaming application from scratch.

The goal of this lecture is not only to write the code, but also to understand how a professional streaming application is designed using best practices.

We are going to simulate a retail organization where invoices are continuously generated from different sales channels.

Those invoices may come from:

* Mobile applications
* Online web portals
* Physical retail stores
* POS systems

All these invoices are collected into a central system.

We are not focusing on how invoices are generated.

Our focus starts from the moment invoice files arrive into a landing zone.

---

# Business Requirement

The requirement is very straightforward.

As soon as invoice files arrive in the landing zone:

1. Read the invoice files continuously
2. Create a raw streaming DataFrame
3. Apply transformation logic
4. Flatten nested invoice structure
5. Write final processed records into a target table

And all this processing should happen continuously as a streaming pipeline.

So essentially:

```text
Landing Zone → Stream Read → Transform → Flatten → Write to Table
```

---

# Understanding the Input Data

The input files are JSON files.

Each JSON record represents one invoice.

Example structure:

```json
{
  "InvoiceNumber": "INV1001",
  "CreatedTime": 123456789,
  "StoreID": "S1",
  "PosID": "P10",
  "CashierID": "C101",
  "CustomerType": "MEMBER",
  "PaymentMethod": "CARD",
  "DeliveryType": "HOME-DELIVERY",

  "DeliveryAddress": {
    "AddressLine": "Street 1",
    "City": "Mumbai",
    "State": "MH",
    "PinCode": "400001",
    "ContactNumber": "9999999999"
  },

  "InvoiceLineItems": [
    {
      "ItemCode": "I100",
      "ItemDescription": "Pan",
      "ItemPrice": 200,
      "ItemQty": 2,
      "TotalValue": 400
    }
  ]
}
```

---

# Expected Output

The final output should be flattened.

Meaning:

If one invoice contains 3 line items, then the output table should contain 3 records.

Each row represents:

* Common invoice information
* Delivery address details
* One individual line item

Example:

| InvoiceNumber | StoreID | ItemCode | Qty | Total |
| ------------- | ------- | -------- | --- | ----- |
| INV1001       | S1      | I100     | 2   | 400   |
| INV1001       | S1      | I101     | 1   | 250   |
| INV1001       | S1      | I102     | 3   | 900   |

So we are essentially converting nested JSON into a denormalized flat structure.

---

# Designing the Streaming Application

We will follow proper software engineering practices.

Instead of writing everything in one notebook cell, we will:

* Create a class
* Organize logic into functions
* Separate responsibilities
* Keep code reusable and testable

---

# Step 1 — Create the Class

```python
class InvoiceStream:

    def __init__(self):
        self.basePath = "/mnt/demo/"
```

---

# Why Create a Class?

A class helps us:

* Organize code
* Reuse methods
* Write test cases easily
* Maintain production-grade code

The constructor initializes commonly used paths.

---

# Step 2 — Define the Schema

## Why Explicit Schema?

Never depend on automatic schema inference in streaming applications.

Why?

Because schema inference:

* Slows down streaming
* Is unreliable
* Can break with malformed files
* Causes inconsistent behavior

Always define schema explicitly.

---

# Creating the Schema Function

```python
def get_schema(self):

    return """
        InvoiceNumber STRING,
        CreatedTime BIGINT,
        StoreID STRING,
        PosID STRING,
        CashierID STRING,
        CustomerType STRING,
        CustomerCardNo STRING,
        TotalAmount DOUBLE,
        NumberOfItems INT,
        PaymentMethod STRING,
        TaxableAmount DOUBLE,
        CGST DOUBLE,
        SGST DOUBLE,
        CESS DOUBLE,
        DeliveryType STRING,

        DeliveryAddress STRUCT<
            AddressLine STRING,
            City STRING,
            State STRING,
            PinCode STRING,
            ContactNumber STRING
        >,

        InvoiceLineItems ARRAY<
            STRUCT<
                ItemCode STRING,
                ItemDescription STRING,
                ItemPrice DOUBLE,
                ItemQty INT,
                TotalValue DOUBLE
            >
        >
    """
```

---

# Understanding Complex Data Types

Notice two complex data types:

## 1. STRUCT

```python
DeliveryAddress STRUCT<...>
```

Used for nested objects.

---

## 2. ARRAY OF STRUCT

```python
InvoiceLineItems ARRAY<STRUCT<...>>
```

This means:

* One invoice contains multiple line items
* Each line item is itself a structure

This is very common in JSON datasets.

---

# Step 3 — Read Streaming Data

Now we will create a streaming DataFrame.

```python
def read_invoices(self):

    return (
        spark.readStream
            .format("json")
            .schema(self.get_schema())
            .load(f"{self.basePath}/data/invoices")
    )
```

---

# Important Concepts

## spark.readStream

Used for continuous streaming ingestion.

---

## .schema()

Explicit schema improves:

* Performance
* Stability
* Predictability

---

## .load()

Reads continuously from landing zone.

Whenever new files arrive:

* Spark detects them
* Micro-batch starts automatically

---

# Step 4 — Explode Invoice Line Items

Now comes the most important transformation.

Remember:

One invoice contains multiple line items.

We want:

```text
1 Invoice → Multiple Rows
```

To achieve this, we use `explode()`.

---

# Explode Function

```python
from pyspark.sql.functions import explode
```

---

# Creating explode_invoices()

```python
def explode_invoices(self, invoice_df):

    return (
        invoice_df.selectExpr(
            "InvoiceNumber",
            "CreatedTime",
            "StoreID",
            "PosID",
            "CustomerType",
            "PaymentMethod",
            "DeliveryType",

            "DeliveryAddress.City",
            "DeliveryAddress.State",
            "DeliveryAddress.PinCode",

            "explode(InvoiceLineItems) as LineItem"
        )
    )
```

---

# What Happens Here?

Suppose input is:

```text
Invoice
 ├── Item1
 ├── Item2
 └── Item3
```

After explode:

```text
Row1 → Item1
Row2 → Item2
Row3 → Item3
```

Each line item becomes an independent row.

---

# Step 5 — Flatten the Struct

After explode:

We still have:

```python
LineItem
```

which is a STRUCT.

We must flatten it.

---

# Creating flatten_invoices()

```python
from pyspark.sql.functions import expr

def flatten_invoices(self, exploded_df):

    return (
        exploded_df
            .withColumn(
                "ItemCode",
                expr("LineItem.ItemCode")
            )
            .withColumn(
                "ItemDescription",
                expr("LineItem.ItemDescription")
            )
            .withColumn(
                "ItemPrice",
                expr("LineItem.ItemPrice")
            )
            .withColumn(
                "ItemQty",
                expr("LineItem.ItemQty")
            )
            .withColumn(
                "TotalValue",
                expr("LineItem.TotalValue")
            )
            .drop("LineItem")
    )
```

---

# Why Use expr()?

Because:

```python
LineItem.ItemCode
```

is nested syntax.

`expr()` converts it into a Spark SQL expression.

---

# Result After Flattening

Now the DataFrame becomes completely flat.

Example:

| InvoiceNumber | ItemCode | Qty |
| ------------- | -------- | --- |
| INV1          | P100     | 2   |
| INV1          | P101     | 1   |

Perfect for analytics and reporting.

---

# Step 6 — Write Stream to Delta Table

Now we write the final stream.

```python
def append_invoices(self, flattened_df):

    return (
        flattened_df.writeStream
            .format("delta")
            .option(
                "checkpointLocation",
                f"{self.basePath}/checkpoint/invoices"
            )
            .outputMode("append")
            .toTable("invoice_line_items")
    )
```

---

# Important Streaming Concepts

## 1. Checkpointing

Checkpoint stores:

* Processed file metadata
* Micro-batch progress
* Streaming state
* Fault recovery information

Without checkpointing:

* Spark cannot recover properly
* Duplicate processing may happen

---

# 2. Append Mode

```python
.outputMode("append")
```

Meaning:

Only new rows are inserted.

No overwriting.

Best for incremental streaming pipelines.

---

# 3. Delta Format

```python
.format("delta")
```

Benefits:

* ACID transactions
* Reliability
* Time travel
* Schema evolution
* Better performance

---

# Step 7 — Main Processing Function

Now we stitch everything together.

```python
def process(self):

    print("Starting Invoice Stream")

    invoices_df = self.read_invoices()

    exploded_df = self.explode_invoices(invoices_df)

    flattened_df = self.flatten_invoices(exploded_df)

    query = self.append_invoices(flattened_df)

    print("Streaming Started")

    return query
```

---

# Streaming Flow

```text
Read Stream
    ↓
Explode Array
    ↓
Flatten Struct
    ↓
Write Stream
```

---

# Complete End-to-End Architecture

```text
JSON Files
    ↓
Landing Zone
    ↓
Spark ReadStream
    ↓
Schema Applied
    ↓
Explode InvoiceLineItems
    ↓
Flatten Struct Columns
    ↓
WriteStream
    ↓
Delta Table
```

---

# Writing Test Suite

Now we test the application properly.

Professional streaming applications must always be tested.

---

# Test Suite Class

```python
class InvoiceStreamTest:

    def __init__(self):
        self.basePath = "/mnt/demo/"
```

---

# Cleaning Environment

```python
def clean_test(self):

    spark.sql("DROP TABLE IF EXISTS invoice_line_items")

    dbutils.fs.rm(
        f"{self.basePath}/checkpoint/",
        True
    )

    dbutils.fs.rm(
        f"{self.basePath}/data/invoices/",
        True
    )

    dbutils.fs.mkdirs(
        f"{self.basePath}/data/invoices/"
    )
```

---

# Why Cleanup?

Because streaming applications maintain state.

We must clean:

* Tables
* Checkpoints
* Landing zones

Otherwise old state affects tests.

---

# Data Ingestion Function

```python
def ingest_data(self, file_name):

    dbutils.fs.cp(
        f"{self.basePath}/test-data/{file_name}",
        f"{self.basePath}/data/invoices/"
    )
```

---

# What Happens Here?

We simulate real-time arrival of files.

Copying file into landing zone triggers streaming micro-batch.

---

# Validation Function

```python
def assert_result(self, expected_count):

    actual_count = spark.table(
        "invoice_line_items"
    ).count()

    assert actual_count == expected_count
```

---

# Waiting for Micro-Batch

```python
import time

def wait_for_microbatch(self):

    time.sleep(30)
```

Streaming takes time.

We must wait for:

* Reading
* Processing
* Writing

before validating results.

---

# Running End-to-End Tests

```python
def run_tests(self):

    self.clean_test()

    stream = InvoiceStream()

    query = stream.process()

    self.ingest_data("invoice1.json")

    self.wait_for_microbatch()

    self.assert_result(1249)

    self.ingest_data("invoice2.json")

    self.wait_for_microbatch()

    self.assert_result(2506)

    query.stop()
```

---

# Understanding the Testing Flow

## Step 1

Start stream.

---

## Step 2

Ingest file into landing zone.

---

## Step 3

Streaming query detects new file.

---

## Step 4

Micro-batch processes file.

---

## Step 5

Validate output table.

---

# Common Errors During Development

## Error 1 — Schema Syntax Error

Example:

```python
STRUCT<...>
ARRAY<...>
```

Missing commas are very common.

---

# Error 2 — outputMode Typo

Wrong:

```python
.outputmode()
```

Correct:

```python
.outputMode()
```

Python is case-sensitive.

---

# What You Learned in This Lecture

In this lecture you learned:

* Real-world Spark Structured Streaming design
* Reading JSON streams
* Explicit schema definition
* Handling nested JSON
* Using STRUCT and ARRAY
* Exploding arrays
* Flattening nested data
* Writing streams into Delta tables
* Using checkpoints
* Streaming test suite creation
* Simulating real-time ingestion
* Validating micro-batch outputs

---

# Most Important Learning

Spark Structured Streaming is fundamentally:

```text
Incremental Processing
```

Every micro-batch processes:

```text
ONLY NEW DATA
```

You do not manually track incremental logic.

Spark handles it automatically using:

* Streaming Query
* Checkpoint
* Micro-batches

---

# Final Mental Model

Always remember this pipeline:

```text
New File Arrives
        ↓
Streaming Query Detects It
        ↓
Micro-Batch Starts
        ↓
Read New File
        ↓
Apply Transformations
        ↓
Write Result
        ↓
Update Checkpoint
        ↓
Wait for Next File
```

That is the heart of Spark Structured Streaming.

---

# Conclusion

Excellent.

Now you are starting to build real production-grade streaming systems.

This lecture covered:

* Architecture
* Transformation logic
* Streaming write
* Checkpointing
* Testing
* Error debugging

In upcoming lectures, we will go deeper into:

* Watermarking
* Stateful streaming
* Window aggregations
* Stream joins
* Fault tolerance
* Performance tuning

Keep practicing.

Keep building.

And keep growing.


Handling complex, nested JSON at scale requires a move away from manual column selection toward a structured, "Medallion Architecture" approach. When you're dealing with thousands of files, the goal is to minimize manual schema definition and maximize Spark's ability to parallelize.

Here is how you can handle this efficiently.

### 1. Ingesting at Scale: Auto Loader

If you are working in a cloud environment (like Databricks), use **Auto Loader** (`cloudFiles`). It is significantly more efficient than a standard `readStream` because it uses file notification services rather than listing the entire directory, which becomes a bottleneck as the number of files grows.

```python
# Scale-optimized ingestion
df = (spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format", "json")
      .option("cloudFiles.schemaLocation", "/path/to/checkpoint/schema") # Schema inference/evolution
      .load("/path/to/raw/json/"))

```

* **Scale Tip:** Use `.option("maxFilesPerTrigger", 1000)` to control backpressure and prevent your cluster from being overwhelmed by a sudden spike in files.

---

### 2. Handling Complex Schemas Dynamically

Instead of manually typing out every `struct` and `explode`, you can use a recursive function to flatten the JSON. This is essential when the JSON structure is deep or changes frequently.

#### The Flattening Logic

To handle `structs` (dot notation) and `arrays` (explode) programmatically, you can iterate through the `df.schema`.

```python
from pyspark.sql.functions import col, explode
from pyspark.sql.types import StructType, ArrayType

def flatten_dataframe(df):
    complex_fields = dict([
        (field.name, field.dataType) 
        for field in df.schema.fields 
        if isinstance(field.dataType, (StructType, ArrayType))
    ])
    
    while len(complex_fields) > 0:
        col_name = list(complex_fields.keys())[0]
        
        # Handle Structs
        if isinstance(complex_fields[col_name], StructType):
            expanded = [col(f"{col_name}.{f.name}").alias(f"{col_name}_{f.name}") 
                        for f in complex_fields[col_name]]
            df = df.select("*", *expanded).drop(col_name)
        
        # Handle Arrays
        elif isinstance(complex_fields[col_name], ArrayType):
            df = df.withColumn(col_name, explode(col(col_name)))
            
        # Re-check for remaining complex fields
        complex_fields = dict([
            (field.name, field.dataType) 
            for field in df.schema.fields 
            if isinstance(field.dataType, (StructType, ArrayType))
        ])
    return df

```

---

### 3. Writing to Delta (The "Bronze to Silver" Step)

Once the data is flattened, write it to a Delta table. Delta is optimized for high-volume streaming and handles "Exactly-once" semantics via checkpoints.

```python
# Apply the flattening logic
flat_df = flatten_dataframe(df)

# Write to Delta
query = (flat_df.writeStream
         .format("delta")
         .outputMode("append")
         .option("checkpointLocation", "/path/to/checkpoint/data")
         .start("/path/to/delta/table"))

```

### High-Scale Considerations

* **Schema Evolution:** JSON sources often change. Use `.option("mergeSchema", "true")` on your Delta sink so that if a new field appears in your JSON files, the Delta table updates automatically without crashing the stream.
* **Trigger Intervals:** For "many files," using `.trigger(availableNow=True)` (or `once=True` in older versions) allows you to process data in micro-batches, which is often more cost-effective than a 24/7 continuous stream if real-time latency isn't required.
* **Partitioning:** If you are filtering by date or a specific ID frequently, ensure your Delta table is partitioned (e.g., `.partitionBy("processing_date")`) to keep query performance high as the table grows into the terabytes.

How deep does your JSON nesting go? If you have nested arrays within nested arrays, we might need to look at specific "watermarking" to handle the state of those explodes.


![image_1778471669324.png](./image_1778471669324.png "image_1778471669324.png")

# Spark Structured Streaming — Streaming Triggers (Professor Style Notes)

Welcome back.

In this lecture, we are going to learn one of the most important concepts in Spark Structured Streaming:

# Streaming Triggers

If you understand triggers properly, you will understand:

* how micro-batches are scheduled
* when processing starts
* how often processing happens
* how Spark controls latency vs cost

And this concept is extremely important in real-world streaming applications.

Because different business requirements need different triggering strategies.

---

# Quick Revision — What is a Streaming Query?

In the earlier lectures, we learned that:

A Spark Structured Streaming application creates something called a:

# Streaming Query

The streaming query is responsible for continuously generating:

# Micro-batches

The workflow looks like this:

```text id="z7dq0j"
Micro-batch 1
       ↓
Process Data
       ↓
Micro-batch 2
       ↓
Process Data
       ↓
Micro-batch 3
       ↓
Process Data
```

So the streaming query continuously:

1. Reads new data
2. Creates a micro-batch
3. Processes it
4. Writes results
5. Repeats again

Now the question becomes:

> WHEN should the next micro-batch start?

That is exactly what triggers control.

---

# Types of Triggers in Spark Structured Streaming

Spark Structured Streaming provides 3 trigger types.

# 1. Unspecified Trigger (Default)

# 2. Fixed Interval Trigger (Processing Time Trigger)

# 3. Available Now Trigger (Once Trigger)

We will understand all three carefully.

---

# 1. Unspecified Trigger (Default Trigger)

Let us start with the simplest one.

If you do NOT specify any trigger explicitly:

Spark automatically uses:

# Unspecified Trigger

This is the default behavior.

---

# How Unspecified Trigger Works

In this mode:

* Spark starts one micro-batch
* Processes available data
* As soon as processing completes
* Immediately starts the next micro-batch

No waiting.

No delay.

No pause.

The next micro-batch starts ASAP.

---

# Visual Understanding

```text id="i1h7c6"
Micro-batch 1 → Finished
Immediately start Micro-batch 2
        ↓
Immediately start Micro-batch 3
        ↓
Immediately start Micro-batch 4
```

Spark continuously keeps processing data.

---

# Code Example — Default Trigger

```python
query = (df.writeStream
         .format("delta")
         .option("checkpointLocation", "/checkpoints/orders")
         .toTable("orders_table"))
```

Notice carefully:

There is NO `.trigger()` specified.

That means Spark automatically uses:

# Unspecified Trigger

---

# When Should We Use Default Trigger?

Use it when:

```text id="u1v4lq"
Process data as quickly as possible
```

Examples:

* Real-time dashboards
* Fraud detection
* Live stock market analytics
* Sensor monitoring
* IoT systems

where low latency matters.

---

# Important Understanding

In this trigger:

Spark is always busy.

As soon as one batch finishes:

another starts immediately.

This minimizes latency.

But it also means:

```text id="l0a2lw"
Higher compute usage
```

because the cluster is continuously active.

---

# 2. Fixed Interval Trigger (Processing Time Trigger)

Now let us move to the second trigger type.

This is called:

# Processing Time Trigger

or

# Fixed Interval Trigger

This allows us to specify a fixed delay between micro-batches.

---

# How It Works

Suppose we specify:

```python
.trigger(processingTime="30 seconds")
```

This means:

> Spark should maintain a 30-second interval between consecutive micro-batches.

---

# Example Scenario

Suppose:

* First micro-batch takes 20 seconds
* Trigger interval = 30 seconds

Then Spark behavior becomes:

```text id="8wp46k"
20 sec processing
+ 10 sec waiting
= 30 sec total interval
```

Spark waits the remaining 10 seconds.

Then starts the next micro-batch.

---

# Another Scenario

Suppose:

* Trigger interval = 30 seconds
* Processing takes 35 seconds

Now what happens?

Spark already exceeded the desired interval.

So Spark immediately starts the next micro-batch.

No waiting.

---

# Visual Timeline

## Case 1 — Processing Faster than Interval

```text id="w7vb71"
0 sec   → Batch starts
20 sec  → Batch completes
30 sec  → Next batch starts
```

---

## Case 2 — Processing Slower than Interval

```text id="06ifg3"
0 sec   → Batch starts
35 sec  → Batch completes
35 sec  → Next batch immediately starts
```

---

# Code Example — Fixed Interval Trigger

```python
query = (df.writeStream
         .format("delta")
         .trigger(processingTime="30 seconds")
         .option("checkpointLocation", "/checkpoints/orders")
         .start("/delta/orders"))
```

---

# When Should We Use Fixed Interval Trigger?

Use this when:

```text id="6o1m1s"
You want to COLLECT data first
and then PROCESS it.
```

This is called:

# Collect-and-Process Model

---

# Real-World Understanding

Suppose your business says:

```text id="lhfg1u"
We don't need instant processing.
Refresh every 10 minutes is enough.
```

In that case:

continuously running micro-batches may be wasteful.

Instead:

* collect data for 10 minutes
* then process it together

This improves efficiency.

---

# Real-World Examples

Good use cases:

* Hourly sales reports
* Periodic ETL jobs
* Aggregated analytics
* IoT telemetry batching
* Log processing every few minutes

---

# Mental Model

Think of fixed interval trigger like:

```text id="7ltglw"
Collect Water in a Bucket
↓
Once bucket is full enough
↓
Start processing
```

Instead of reacting to every single drop.

---

# 3. Available Now Trigger (Once Trigger)

Now let us discuss the most interesting trigger type.

# Available Now Trigger

Earlier versions of Spark used:

```text id="fg5mwd"
Trigger.Once()
```

Modern Spark uses:

```text id="xw9n4l"
availableNow=True
```

---

# What Does Available Now Mean?

This trigger says:

```text id="2pvhv0"
Process ALL currently available data
AND THEN STOP.
```

This is extremely important.

Unlike normal streaming queries:

the query does NOT keep running forever.

It automatically shuts down after processing available data.

---

# Visual Understanding

```text id="d4wixn"
Read all available files
       ↓
Process everything
       ↓
Write results
       ↓
STOP streaming query
```

---

# Why Is This Useful?

Let us understand a real-world scenario.

Suppose your requirement is:

```text id="q96a8h"
Process data once every hour
```

Now imagine using fixed interval trigger.

What happens?

* Job processes for 2 minutes
* Then sits idle for 58 minutes

Meanwhile:

* Cluster is still alive
* Compute resources still allocated
* Money still being spent

Very wasteful.

---

# Available Now Solves This

Instead:

you can:

1. Start job every hour using scheduler
2. Process all available data
3. Stop job automatically
4. Release cluster/resources

Huge cost savings.

---

# Code Example — Available Now Trigger

```python
query = (df.writeStream
         .format("delta")
         .trigger(availableNow=True)
         .option("checkpointLocation", "/checkpoints/orders")
         .start("/delta/orders"))
```

---

# Important Understanding

This behaves like:

# Batch Processing

because:

* job starts
* processes all data
* stops

BUT internally:

Spark still uses:

* streaming APIs
* checkpointing
* incremental processing

This is extremely powerful.

---

# Incremental Batch Processing

This is one of the biggest advantages.

Available Now allows you to build:

# Incremental Batch Pipelines

without writing complicated logic manually.

Spark automatically remembers:

* what was already processed
* what is new

using checkpoints.

---

# Real-World Architecture

Example:

```text id="m2u10s"
Scheduler (every 1 hour)
        ↓
Start Streaming Job
        ↓
Process New Data Only
        ↓
Write Results
        ↓
Stop Automatically
```

This gives:

* Batch-like behavior
* Streaming-like intelligence

Best of both worlds.

---

# When Should We Use Available Now Trigger?

Perfect for:

* Hourly ETL jobs
* Daily ingestion pipelines
* Cost-sensitive streaming workloads
* Incremental batch architectures
* Scheduled streaming jobs

---

# Comparing All Three Triggers

# 1. Unspecified Trigger

## Behavior

```text id="v7v8s0"
Process ASAP continuously
```

## Best For

* Real-time systems
* Low latency analytics

---

# 2. Fixed Interval Trigger

## Behavior

```text id="im1x5w"
Wait fixed interval between batches
```

## Best For

* Periodic processing
* Collect-and-process workloads

---

# 3. Available Now Trigger

## Behavior

```text id="cshv4q"
Process all available data once and stop
```

## Best For

* Incremental batch pipelines
* Cost optimization

---

# Trigger Comparison Table

| Trigger Type   | Query Runs Continuously? | Wait Between Batches? | Stops Automatically? | Best Use Case          |
| -------------- | ------------------------ | --------------------- | -------------------- | ---------------------- |
| Unspecified    | Yes                      | No                    | No                   | Real-time processing   |
| Fixed Interval | Yes                      | Yes                   | No                   | Periodic micro-batches |
| Available Now  | No                       | N/A                   | Yes                  | Incremental batch      |

---

# Key Performance Insight

Choosing the correct trigger is NOT just a technical decision.

It directly impacts:

* latency
* throughput
* compute usage
* cloud cost

---

# Mental Model to Remember Forever

# Unspecified Trigger

```text id="8l03hc"
Keep running continuously
```

---

# Fixed Interval Trigger

```text id="85e8ko"
Wait → Process → Wait → Process
```

---

# Available Now Trigger

```text id="jlwm0z"
Start → Process Everything → Stop
```

---

# Important Interview Question

## Why is Available Now Trigger Powerful?

Because it combines:

```text id="wdm5zj"
Streaming Incremental Processing
+
Batch-style Resource Efficiency
```

This is one of the most important concepts in modern data engineering.

---

# Final Takeaway

The trigger controls:

```text id="r7d5tb"
WHEN Spark should create the next micro-batch
```

Understanding triggers helps you design:

* scalable systems
* low-cost systems
* low-latency systems
* efficient pipelines

correctly.

---

# End of Lecture

That’s all for this lecture.

In the next lecture, we will implement practical examples using:

# Available Now Trigger

so you can understand how to build incremental batch pipelines using streaming APIs.

Keep learning and keep growing.


![image_1778472219443.png](./image_1778472219443.png "image_1778472219443.png")

# Spark Structured Streaming as Unified Batch + Streaming Engine

Welcome back.

In this lecture, we are going to learn one of the most powerful capabilities of Spark Structured Streaming.

Spark allows us to write **one unified application** that can work in:

* **Batch mode**
* **Streaming mode**

using the **same codebase**.

That means:

* No duplicate applications
* No separate batch pipeline
* No separate streaming pipeline
* No major code changes

The exact same Spark Structured Streaming APIs can execute both workloads.

This is one of the biggest architectural advantages of Structured Streaming.

---

# The Big Idea

Traditionally, systems used:

* One engine for batch processing
* Another engine for stream processing

Which created problems:

* Duplicate logic
* Duplicate testing
* Higher maintenance
* Different behaviors
* More bugs

Spark Structured Streaming solves this problem by introducing a **unified execution model**.

You write your logic once.

Then Spark decides whether to execute it:

* continuously as a stream
* or once as a batch

depending on the trigger configuration.

---

# Our Existing Invoice Streaming Example

In earlier lectures, we created an invoice streaming application.

The architecture looked like this:

```text
Landing Zone (JSON invoice files)
        ↓
Read Stream
        ↓
Raw DataFrame
        ↓
Explode Invoice Items
        ↓
Flatten Structure
        ↓
Write into Target Table
```

---

# Input Data

We receive invoice files in JSON format.

Each file contains multiple invoices.

Each invoice contains:

* invoice-level information
* multiple line items

Example structure:

```json
{
  "InvoiceNumber": "INV001",
  "StoreID": "S1",
  "CustomerType": "Member",
  "InvoiceLineItems": [
      {...},
      {...}
  ]
}
```

---

# What Was Our Goal Earlier?

Earlier, our application worked only as a **streaming application**.

Meaning:

* query continuously runs
* keeps checking landing zone
* processes newly arriving files

Now we want something more powerful.

We want:

# Goal

Create one application that can:

## Option 1 — Run as Batch

Process all currently available data once and stop.

OR

## Option 2 — Run as Continuous Stream

Keep processing incoming files continuously.

And we want this without changing business logic.

---

# Important Observation

Our business logic is NOT changing.

We still want to:

* read invoices
* explode line items
* flatten records
* write output

So:

# Business Logic Remains Same

The only thing changing is:

# HOW the query gets triggered

That means:

* read logic stays same
* transformation stays same
* schema stays same

Only the **write stream trigger** changes.

---

# Creating New Notebook

We create a new notebook:

```python
StreamingBatch
```

This notebook will contain a unified application.

---

# Reusing Old Code

We copy our old streaming application.

Original class:

```python
InvoiceStream
```

New class:

```python
InvoiceStreamBatch
```

---

# What Does NOT Change?

The following components remain exactly same:

## Constructor

No change.

---

## Schema

No change.

We still read same invoice JSON files.

---

## Read Logic

No change.

Still reading from landing zone.

---

## Transformation Logic

No change.

Still:

* exploding invoice items
* flattening records

because business requirement is identical.

---

# Where Do We Need Changes?

Only here:

# appendInvoices()

Why?

Because this is where:

```python
writeStream
```

is defined.

And triggers are configured during streaming write.

---

# Original appendInvoices()

Earlier version:

```python
appendInvoices(flattenedDF)
```

Now we modify it.

---

# New appendInvoices()

We add a trigger parameter.

```python
appendInvoices(flattenedDF, trigger="batch")
```

---

# Why Add Trigger Parameter?

Because now this method should support:

* batch execution
* streaming execution

depending on trigger type.

---

# Default Trigger = Batch

We set:

```python
trigger="batch"
```

Meaning:

If nothing is specified,
run application in batch mode.

---

# Important Refactoring

Originally we probably had:

```python
.writeStream
...
.start()
```

inside the query definition.

But now we separate:

# Query Definition

from

# Query Execution

---

# Why?

Because we want to:

1. define query
2. configure trigger
3. then start query

This gives flexibility.

---

# Query is Lazy Until Action

Important concept.

This part:

```python
.writeStream
.format()
.option()
.outputMode()
```

ONLY defines query.

It does NOT start execution.

Execution starts only when:

```python
.start()
```

is called.

Exactly like Spark transformations vs actions.

---

# Building Query First

So now we store query temporarily.

```python
query = flattenedDF.writeStream...
```

without calling `.start()` yet.

---

# Adding Conditional Trigger Logic

Now we check:

```python
if trigger == "batch":
```

---

# Batch Mode Trigger

In batch mode we configure:

```python
.trigger(availableNow=True)
```

This is extremely important.

---

# What is AvailableNow Trigger?

The Available Now trigger means:

# Process Everything Currently Available

AND

# Automatically Stop

---

# Behavior of AvailableNow

Suppose landing zone already has:

```text
10 files
```

before query starts.

Then Spark will:

* process all 10 files
* complete all required micro-batches
* stop automatically

---

# Important Clarification

AvailableNow is still:

# Structured Streaming

It still uses:

* micro-batches
* checkpoints
* streaming engine

But behaves like batch execution.

---

# Why Is This Powerful?

Because now we get:

* incremental processing
* exactly-once guarantees
* checkpointing
* fault tolerance

while still running like batch.

---

# Batch Trigger Code

```python
query.trigger(availableNow=True).start()
```

---

# Else Condition → Streaming Mode

Now we handle continuous streaming.

```python
else:
```

---

# Processing Time Trigger

We configure:

```python
.trigger(processingTime=trigger)
```

---

# Example

If user passes:

```python
"30 seconds"
```

then Spark creates:

# Continuous Micro-Batches

every 30 seconds.

---

# Meaning of Processing Time Trigger

Spark will:

1. wait for trigger interval
2. check for new data
3. process micro-batch
4. repeat forever

until query is manually stopped.

---

# Final Return

Both branches finally return:

```python
.start()
```

---

# Summary of Trigger Logic

| Trigger Value  | Behavior                             |
| -------------- | ------------------------------------ |
| `"batch"`      | Process available data once and stop |
| `"30 seconds"` | Continuous streaming every 30s       |

---

# Additional Important Option

Now professor introduces another critical option.

```python
maxFilesPerTrigger
```

Example:

```python
.option("maxFilesPerTrigger", 1)
```

---

# What Does maxFilesPerTrigger Do?

It limits how many files one micro-batch can process.

---

# Example Scenario

Suppose landing zone contains:

```text
10 files
```

and:

```python
maxFilesPerTrigger = 1
```

Then:

| Micro-Batch | Files Processed |
| ----------- | --------------- |
| Batch 1     | File 1          |
| Batch 2     | File 2          |
| Batch 3     | File 3          |

and so on.

---

# Why Is This Useful?

Because sometimes we want predictable processing time.

Example goal:

```text
Each micro-batch should finish within 10 seconds
```

If one batch processes too many files:

* latency increases
* memory pressure increases
* instability increases

So we limit input size.

---

# Default Value

Default:

```text
1000 files
```

Meaning one micro-batch can process up to 1000 files.

---

# Important Real-World Insight

Usually streaming systems receive:

* many small files

So default value works fine.

But for huge files or heavy transformations,
we often reduce this number.

---

# Extremely Important Understanding

Now comes a subtle but critical concept.

---

# AvailableNow + maxFilesPerTrigger

Suppose:

```text
10 files available
maxFilesPerTrigger = 1
trigger = availableNow
```

What happens?

---

# Spark Behavior

Spark will create:

```text
10 separate micro-batches
```

because each micro-batch can process only 1 file.

But:

# Spark WILL NOT STOP

until ALL currently available files are processed.

---

# Important Point

AvailableNow means:

# Process ALL data currently available

even if multiple micro-batches are required.

---

# But What About New Incoming Files?

Suppose while processing:

* new files arrive later

Will they be processed?

# No.

AvailableNow only processes:

# Data available at query start time

Then query stops.

---

# Why Is It Called “Available Now”?

Because Spark says:

```text
Whatever is available now,
process everything.
```

Then terminate.

---

# Updating process() Method

Now we move to:

```python
process()
```

This is public method called externally.

Earlier:

```python
process()
```

Now:

```python
process(trigger="batch")
```

---

# Why?

Because process() should forward trigger type to:

```python
appendInvoices()
```

---

# Final Unified Architecture

Now our application supports:

| Mode      | Trigger        |
| --------- | -------------- |
| Batch     | availableNow   |
| Streaming | processingTime |

using SAME code.

---

# Running Application

We connect cluster and execute.

No errors.

Meaning:

* syntax correct
* trigger logic correct

---

# Next Step — Testing

Now professor rewrites test suite.

Because application now supports:

1. streaming mode
2. batch mode

So both should be tested separately.

---

# Creating New Test Suite

New notebook:

```python
StreamingBatchTestSuite
```

---

# Reusing Existing Tests

Most test logic remains same.

Because business logic did not change.

---

# What Stays Same?

These remain unchanged:

* constructor
* cleanup
* data ingestion
* assertions
* wait logic

because transformations remain same.

---

# Stream Test Case

New method:

```python
runStreamTest()
```

---

# Running in Streaming Mode

We instantiate:

```python
InvoiceStreamBatch
```

Then call:

```python
process("30 seconds")
```

Meaning:

* continuous stream
* micro-batch every 30 seconds

---

# Test Flow

For each iteration:

1. ingest file
2. wait for micro-batch
3. validate output

Finally:

```python
query.stop()
```

because continuous streaming never stops automatically.

---

# Batch Test Case

Now comes different logic.

Method:

```python
runBatchTest()
```

---

# Key Difference

In streaming mode:

```text
Start query first
Then ingest data
```

because query keeps listening continuously.

---

# But Batch Mode Is Different

If we start batch query first:

* it sees no data
* finishes immediately
* stops

So batch test must:

# Ingest Data First

THEN

# Start Query

---

# Batch Test Flow

## Step 1

Cleanup.

---

## Step 2

Instantiate application.

---

## Step 3

Ingest files first.

Example:

```text
2 files
```

---

## Step 4

Start batch query.

```python
process("batch")
```

---

# What Happens Internally?

Because:

```python
availableNow=True
maxFilesPerTrigger=1
```

Spark creates:

| Micro-Batch | File   |
| ----------- | ------ |
| 1           | File 1 |
| 2           | File 2 |

Then query automatically stops.

---

# Validation

After waiting:

assert expected record count.

---

# Second Iteration

Now ingest third file.

Then:

* restart query
* process new file
* validate again

---

# Important Observation

In batch mode:

# Query Stops Automatically

So we must restart query every time new data arrives.

---

# Unlike Continuous Streaming

Continuous stream remains active.

Batch execution does not.

---

# Final Result

Both tests pass successfully:

* stream mode
* batch mode

---

# Core Learning of This Lecture

This lecture teaches one of the most important Spark Structured Streaming concepts:

# Unified Batch + Streaming Architecture

---

# Final Mental Model

Think like this:

| Component             | Changes? |
| --------------------- | -------- |
| Schema                | No       |
| Read Logic            | No       |
| Business Logic        | No       |
| Transformations       | No       |
| Write Destination     | No       |
| Trigger Configuration | YES      |

---

# Key Spark Structured Streaming Concepts Learned

## 1. Unified Codebase

Same code supports:

* batch
* streaming

---

## 2. AvailableNow Trigger

Processes all currently available data and stops automatically.

---

## 3. Processing Time Trigger

Runs continuously at fixed intervals.

---

## 4. maxFilesPerTrigger

Limits number of files per micro-batch.

---

## 5. Query Definition vs Execution

`.writeStream` defines query.

`.start()` executes query.

---

## 6. Batch Mode Still Uses Streaming Engine

Even batch execution internally uses:

* micro-batches
* checkpoints
* incremental processing

---

# Real Industry Importance

This architecture is heavily used in modern data engineering because it allows:

* reusable pipelines
* simpler maintenance
* lower development effort
* unified testing
* incremental ETL processing
* scalable ingestion systems

This is one of the major reasons Spark Structured Streaming became extremely popular in enterprise data engineering systems.

See you again in the next lecture.

Keep learning and keep growing.


![image_1778474299306.png](./image_1778474299306.png "image_1778474299306.png")

# Spark Structured Streaming Sources and Sinks

Welcome back.

In this lecture, we are going to learn about one of the most important foundational concepts in Spark Structured Streaming:

# Sources and Sinks

If you understand sources and sinks properly, you will understand how data flows through a streaming system.

This lecture is extremely important because every streaming application is built using:

```text
Source → Processing → Sink
```

So let us understand everything step by step.

---

# Big Picture of Spark Structured Streaming

Before understanding sources and sinks, let us quickly revise how a Spark Structured Streaming application works internally.

When we start a Spark Structured Streaming application:

```python
readStream → transformations → writeStream
```

Spark creates a background execution engine.

This background engine is known as:

# Streaming Query

---

# What Does Streaming Query Do?

The Streaming Query is responsible for:

* continuously monitoring data
* creating micro-batches
* reading incremental data
* processing data
* writing results

Think of it as the brain of the streaming application.

---

# Internal Architecture

The flow looks like this:

```text
Streaming Source
       ↓
Read Stream
       ↓
Micro-Batch Processing
       ↓
Transformations
       ↓
Write Stream
       ↓
Streaming Sink
```

---

# Important Internal Mechanism

Spark Structured Streaming internally works using:

# Micro-Batches

Meaning:

Instead of processing one event at a time,
Spark groups incoming data into small batches.

Each micro-batch contains:

* newly arrived data
* configured offsets/files
* processing information

Then Spark executes the transformations on that batch.

---

# What is a Source?

A source is:

# The place from where Spark reads streaming data

In simple words:

A source continuously provides incremental data to Spark.

---

# Real-World Analogy

Think of a source like a water pipeline feeding water into a machine.

The machine processes water and sends it somewhere else.

Similarly:

```text
Source → Spark → Sink
```

---

# What is a Sink?

A sink is:

# The destination where Spark writes processed streaming data

After Spark finishes processing:

* results
* transformed data
* aggregated outputs

are written to the sink.

---

# Important Understanding

Every streaming application needs BOTH:

| Component | Purpose    |
| --------- | ---------- |
| Source    | Read data  |
| Sink      | Write data |

Without source → no input.

Without sink → nowhere to save results.

---

# Streaming Source Example We Already Used

In earlier lectures, we already worked with one streaming source.

That source was:

# Directory / Landing Zone

---

# Example

We had a folder like:

```text
/landing-zone/invoices/
```

New JSON invoice files arrived continuously.

Spark used:

```python
spark.readStream
```

to continuously monitor the directory.

---

# What Happens Internally?

Whenever new files arrive:

```text
invoice1.json
invoice2.json
invoice3.json
```

Spark detects them.

Then the streaming query creates a micro-batch.

Then readStream reads only newly arrived files.

---

# Important Point

Spark Structured Streaming processes:

# Incremental Data

NOT entire dataset repeatedly.

---

# Sources Supported by Spark Structured Streaming

Now the professor explains all major sources supported by Spark.

The list is actually small but very powerful.

---

# 1. File Source / Directory Source

This is the most commonly used source.

Also called:

* file source
* landing zone source
* directory source

---

# How It Works

Spark continuously monitors a directory.

Whenever new files arrive:

* CSV
* JSON
* Parquet
* Avro

Spark processes them incrementally.

---

# Example

```python
df = spark.readStream \
    .format("json") \
    .load("/landing-zone/")
```

---

# Why Is File Source Popular?

Because most enterprise systems exchange data using files.

Example architecture:

```text
Application
    ↓
Landing Zone
    ↓
Spark Streaming
```

---

# Real Industry Usage

File source is heavily used in:

* ETL pipelines
* Data lakes
* Batch ingestion systems
* Cloud storage ingestion
* Incremental file processing

---

# Important Characteristic

Spark treats newly arriving files as streaming events.

Even though files are static,
their arrival pattern is continuous.

That makes them behave like streams.

---

# 2. Delta Table as Streaming Source

Now comes a very powerful feature.

If Spark is integrated with Delta Lake:

# Delta Tables can also act as streaming sources

---

# What Does This Mean?

Spark can read:

# Incremental changes from Delta tables

instead of only reading files.

---

# Example

```python
spark.readStream.table("sales_delta")
```

or

```python
spark.readStream.format("delta")
```

---

# Why Is This Powerful?

Because Delta Lake maintains:

* transaction logs
* version history
* ACID guarantees

Spark can use these logs to identify:

# Newly inserted data

and process only incremental records.

---

# Real-World Usage

This is very common in:

* Medallion architecture
* Bronze → Silver → Gold pipelines
* Incremental ETL systems
* CDC pipelines

---

# Example Architecture

```text
Bronze Delta Table
        ↓
Streaming Read
        ↓
Transformations
        ↓
Silver Delta Table
```

---

# Important Insight

Delta Table can work as BOTH:

| Role   | Supported? |
| ------ | ---------- |
| Source | Yes        |
| Sink   | Yes        |

This makes Delta Lake extremely powerful for streaming pipelines.

---

# 3. Kafka as Streaming Source

Spark also provides native support for:

# Apache Kafka

---

# What is Kafka?

Kafka is a distributed event streaming platform.

It is widely used for:

* real-time messaging
* event ingestion
* log streaming
* IoT streaming
* real-time analytics

---

# Spark + Kafka Integration

Spark can directly consume Kafka topics using:

```python
readStream
```

---

# Example

```python
spark.readStream \
    .format("kafka")
```

---

# What Can Spark Read?

Spark can read:

* messages
* events
* logs
* JSON payloads
* Avro events

incrementally from Kafka topics.

---

# Real-World Example

```text
Applications → Kafka → Spark Streaming → Analytics
```

---

# Why Kafka Is Important?

Kafka is one of the most commonly used streaming systems in the industry.

Many real-time systems are built using:

* Kafka
* Spark Streaming
* Delta Lake

together.

---

# 4. External Source Connectors

Professor now explains something important.

Spark only provides limited connectors out of the box.

But many external vendors provide additional connectors.

---

# What Are External Connectors?

External connectors allow Spark to connect to systems like:

* databases
* cloud services
* messaging systems
* SaaS applications

---

# Examples

Different vendors create connectors for:

* MongoDB
* Cassandra
* Snowflake
* Kinesis
* Azure Event Hub
* Google Pub/Sub

and many others.

---

# Role of Vendors

Third-party vendors develop connectors so Spark can read incremental data from their systems.

---

# Databricks Contribution

Professor mentions that Databricks also develops many integrations and connectors for popular enterprise systems.

This makes Spark ecosystem extremely extensible.

---

# Summary of Streaming Sources

| Source Type         | Purpose                        |
| ------------------- | ------------------------------ |
| File / Directory    | Read arriving files            |
| Delta Table         | Read incremental table changes |
| Kafka               | Read real-time events/messages |
| External Connectors | Read from external systems     |

---

# Now Let Us Learn About Sinks

After processing data,
Spark must write results somewhere.

That destination is called:

# Sink

---

# What is a Sink?

A sink is:

# The final destination of processed streaming data

---

# Sink Flow

```text
Source
   ↓
Processing
   ↓
Sink
```

---

# Streaming Sinks Supported by Spark

Spark supports several sinks.

Let us understand each one carefully.

---

# 1. Delta Table Sink

This is the sink we already used earlier.

Example:

```python
writeStream.toTable("invoice_line_items")
```

---

# Why Delta Sink Is Popular?

Because Delta Lake provides:

* ACID transactions
* reliability
* scalability
* schema enforcement
* fault tolerance

---

# Real Usage

Most modern Databricks streaming pipelines write into Delta tables.

---

# Example Architecture

```text
Kafka
   ↓
Spark Streaming
   ↓
Delta Table
```

---

# 2. File Sink / Directory Sink

Spark can also write processed data into files.

Meaning:

writeStream can continuously create output files.

---

# Example

```python
writeStream \
    .format("parquet")
```

---

# What Happens?

Spark writes:

* Parquet files
* JSON files
* CSV files

into a target directory.

---

# Important Understanding

This target directory itself may later become another system’s source.

---

# Example Pipeline

```text
Source Files
    ↓
Spark Processing
    ↓
Output Files
    ↓
Consumed by Another System
```

---

# Real Usage

File sinks are commonly used in:

* Data lakes
* Cloud storage
* Archival systems
* Batch interoperability

---

# 3. Kafka Sink

Spark can also write output data into Kafka topics.

Meaning:

Spark becomes a producer.

---

# Example

```python
writeStream \
    .format("kafka")
```

---

# Real-World Usage

Useful for:

* event-driven systems
* real-time notifications
* downstream consumers
* streaming architectures

---

# Example Flow

```text
Kafka → Spark → Kafka
```

This pattern is very common in event processing systems.

---

# 4. foreach Sink

Now comes one of the most powerful sinks.

Professor calls it:

# Swiss Army Knife of Spark Sinks

---

# Why?

Because foreach gives maximum flexibility.

---

# What Does foreach Do?

It allows us to write custom logic for every batch or record.

Meaning:

If Spark does not support some destination system directly,
we can manually write integration logic.

---

# Example Use Cases

Using foreach we can send data to:

* REST APIs
* Custom databases
* Legacy systems
* External applications
* Proprietary systems

---

# Why Is foreach Powerful?

Because it removes connector limitations.

You can integrate Spark with virtually anything.

---

# Important Mental Model

Think of foreach as:

```text
"Custom sink implementation"
```

---

# Real-World Importance

Many enterprise integrations are implemented using:

# foreachBatch

or

# foreach

because companies often use custom internal systems.

---

# Important Insight from Professor

Professor mentions:

# Spark has limited built-in connectors

Most enterprise integrations come through:

* external vendors
* custom implementations
* Databricks integrations
* third-party libraries

---

# Final Architecture View

Now we can visualize a complete streaming system.

```text
Streaming Source
       ↓
Spark Read Stream
       ↓
Micro-Batch Engine
       ↓
Transformations
       ↓
Write Stream
       ↓
Streaming Sink
```

---

# Complete List of Common Sources and Sinks

| Type                | Sources | Sinks     |
| ------------------- | ------- | --------- |
| File System         | Yes     | Yes       |
| Delta Table         | Yes     | Yes       |
| Kafka               | Yes     | Yes       |
| foreach             | No      | Yes       |
| External Connectors | Yes     | Sometimes |

---

# Key Takeaways

# 1. Source

A source provides streaming input data.

---

# 2. Sink

A sink stores or forwards processed data.

---

# 3. Spark Uses Micro-Batches

Streaming Query continuously creates micro-batches.

---

# 4. File Source Is Most Common

Landing zone ingestion is heavily used in industry.

---

# 5. Delta Lake Is Extremely Powerful

Because it supports both:

* source
* sink

---

# 6. Kafka Is Core Streaming Infrastructure

Used heavily for real-time systems.

---

# 7. foreach Is the Most Flexible Sink

Useful for unsupported/custom systems.

---

# Final Mental Model

Always think about Spark Structured Streaming using this architecture:

```text
Read Stream
    ↓
Source
    ↓
Micro-Batch Processing
    ↓
Transformations
    ↓
Write Stream
    ↓
Sink
```

Everything in Spark Structured Streaming revolves around this flow.

See you again in the next lecture.

Keep learning and keep growing.


![image_1778474619435.png](./image_1778474619435.png "image_1778474619435.png")

# Delta Table as Streaming Source & Chaining Streaming Jobs

Welcome back.

In this lecture, we are going to learn two extremely important concepts in Spark Structured Streaming.

# Topics of This Lecture

## 1. Delta Table as a Streaming Source

How to use a Delta table with:

```python id="ewjlwm"
readStream
```

to consume incremental data.

---

## 2. Chaining Streaming Jobs

How to create:

* multiple streaming queries
* multiple streaming applications
* dependent streaming pipelines

where one streaming job feeds another streaming job.

This is one of the most important real-world architectures used in modern Data Engineering systems.

---

# What Have We Done So Far?

Until now, we created:

# Single Streaming Job

Meaning:

```text id="7cyb84"
One source
    ↓
One streaming query
    ↓
One target table
```

Spark created:

# One Streaming Query Thread

which continuously executed micro-batches.

---

# But Real Projects Are Different

In real enterprise systems:

* one streaming job is usually NOT enough
* processing is divided into stages
* multiple jobs work together

This creates:

# Streaming Pipelines

or

# Chained Streaming Jobs

---

# What Does “Chaining Streaming Jobs” Mean?

It means:

One streaming job produces data,
and another streaming job consumes that data.

Example:

```text id="p0qmqh"
Streaming Job 1
      ↓
Produces Table A
      ↓
Streaming Job 2
      ↓
Reads Table A
      ↓
Produces Table B
```

This chaining allows us to build scalable layered architectures.

---

# Real-World Example — Medallion Architecture

Professor now introduces a miniature version of:

# Medallion Architecture

This architecture is extremely common in:

* Data Lake projects
* Lakehouse systems
* Databricks implementations

---

# Medallion Layers

Usually there are three layers:

| Layer  | Purpose                        |
| ------ | ------------------------------ |
| Bronze | Raw ingestion                  |
| Silver | Cleaned/transformed data       |
| Gold   | Business-ready aggregated data |

---

# Earlier Architecture

Previously our invoice pipeline looked like this:

```text id="x00svm"
Landing Zone
      ↓
Read Stream
      ↓
Explode Invoice
      ↓
Flatten Data
      ↓
Final Table
```

Everything happened in:

# One Streaming Job

---

# New Architecture

Now we want to split processing into two independent streaming jobs.

---

# Bronze Layer Job

The first streaming job should only:

* ingest raw files
* store raw data
* create raw ingestion table

Nothing else.

---

# Silver Layer Job

The second streaming job should:

* read bronze table
* explode invoices
* flatten records
* produce final table

---

# Final Pipeline Architecture

```text id="4m6w04"
Landing Zone
      ↓
Bronze Streaming Job
      ↓
Bronze Table
      ↓
Silver Streaming Job
      ↓
Silver Table
```

This is the core concept of chained streaming jobs.

---

# Why Is This Architecture Powerful?

Because each layer becomes:

* independent
* scalable
* reusable
* maintainable

---

# Real Enterprise Benefits

Different teams can own:

| Layer  | Responsibility      |
| ------ | ------------------- |
| Bronze | Ingestion Team      |
| Silver | Transformation Team |
| Gold   | BI/Analytics Team   |

---

# Creating New Notebook

We create a new notebook:

```python id="t8n7yo"
MedallionApproach
```

---

# Reusing Existing Code

Instead of writing everything from scratch,
we copy our previous invoice streaming application.

Because:

# Business Logic Is Mostly Same

---

# Splitting Into Two Classes

Now we divide processing into:

## Bronze Layer Class

and

## Silver Layer Class

---

# Bronze Layer Responsibility

The Bronze class will:

| Step | Action                  |
| ---- | ----------------------- |
| 1    | Read raw invoice files  |
| 2    | Store raw records       |
| 3    | Write into bronze table |

No transformations.

No flattening.

No business logic.

---

# Silver Layer Responsibility

The Silver class will:

| Step | Action             |
| ---- | ------------------ |
| 1    | Read bronze table  |
| 2    | Explode invoices   |
| 3    | Flatten data       |
| 4    | Write silver table |

---

# Bronze Layer Development

Now professor starts modifying Bronze layer.

---

# Constructor Remains Same

No changes.

---

# Schema Remains Same

Still reading same invoice JSON structure.

---

# readInvoices() Remains Mostly Same

Still using:

```python id="g0o0wf"
spark.readStream
```

Still reading JSON files from landing zone.

---

# New Feature — cleanSource

Professor now introduces a very important file source option.

```python id="61g1dm"
cleanSource
```

This works with:

# File / Directory Streaming Sources

---

# What Does cleanSource Do?

It automatically handles files that were already processed.

---

# Why Is This Needed?

Suppose micro-batch 1 processes:

```text id="g7ltq9"
invoice1.json
```

After processing completes,
what should happen to that file?

Options:

* keep it
* delete it
* archive it

Spark provides automation for this.

---

# Option 1 — Delete Processed Files

```python id="x10d9v"
.option("cleanSource", "delete")
```

---

# What Happens?

After next micro-batch starts,
previously processed files are deleted automatically.

---

# Why Is This Dangerous?

Because raw files are valuable.

If something fails later:

* debugging becomes difficult
* replay becomes impossible
* recovery becomes difficult

So deleting raw files is risky.

---

# Better Approach — Archive

Instead of deleting files:

# Move them to Archive Location

---

# Archive Configuration

```python id="c8n0k4"
.option("cleanSource", "archive")
```

---

# But Where Should Files Be Archived?

Spark needs archive location.

---

# sourceArchiveDir

```python id="5n9yv2"
.option("sourceArchiveDir", archive_path)
```

---

# What Happens Internally?

After processing:

```text id="dy2onp"
Landing Zone
      ↓
Processed File
      ↓
Moved to Archive Directory
```

This keeps landing zone clean while preserving raw data history.

---

# Why Is Archival Better?

Because archived files can later be used for:

* replay
* debugging
* reprocessing
* auditing
* compliance

---

# Important Production Insight

Most enterprise systems:

# Archive raw files

instead of deleting them.

---

# Bronze process() Method

Now Bronze layer defines main process method.

---

# Bronze Workflow

```text id="wrpp0u"
Read Landing Zone
      ↓
Write Bronze Table
```

---

# Bronze WriteStream

Bronze layer writes raw invoices into:

```python id="m01h3t"
Invoices_BZ
```

(Bronze table)

---

# Checkpointing

Professor defines checkpoint location.

```python id="m5f95d"
checkpointLocation
```

---

# Why Are Checkpoints Important?

Structured Streaming uses checkpoints for:

* fault tolerance
* offset tracking
* recovery
* exactly-once guarantees

---

# Bronze Query Name

Professor introduces another important feature:

# queryName

---

# Why queryName Matters?

Real systems may run:

```text id="fyszcx"
Hundreds of streaming queries
```

Without names,
monitoring becomes difficult.

---

# Query Naming

```python id="b1t9ix"
.queryName("bronze_ingestion")
```

---

# Why Is This Useful?

In Spark UI,
we can identify:

* which query is running
* what it does
* performance metrics

---

# Bronze Layer Summary

Bronze layer:

| Task       | Description      |
| ---------- | ---------------- |
| Read       | Landing zone     |
| Store      | Raw table        |
| Archive    | Processed files  |
| Query Name | bronze_ingestion |

---

# Now Moving to Silver Layer

Silver layer becomes simpler.

Why?

Because it does NOT read files.

It reads:

# Delta Table

---

# Major Learning — Delta Table as Streaming Source

Now professor introduces the main concept.

---

# Reading Delta Table Using readStream

Instead of:

```python id="lpb7dk"
.load(directory)
```

we use:

```python id="o7dfnr"
.readStream.table("Invoices_BZ")
```

---

# Extremely Important Concept

Spark can treat Delta tables as:

# Streaming Sources

---

# What Does This Mean?

Whenever Bronze layer inserts new data:

Silver layer automatically detects it.

Then Silver layer creates micro-batches for newly inserted rows.

---

# Incremental Delta Processing

Spark reads only:

# Newly added records

from Delta table.

Not entire table repeatedly.

---

# Why Is This Powerful?

Because Delta Lake maintains:

* transaction logs
* metadata
* versions

Spark uses transaction logs to identify new data.

---

# Big Advantage

When reading from Delta tables:

We do NOT need to specify:

* schema
* file format

because Delta already stores metadata.

---

# Compare Both Reads

---

## File Source Read

Needs:

* format
* schema
* path

```python id="zrxzib"
.readStream
.format("json")
.schema(...)
.load(path)
```

---

## Delta Table Read

Needs only:

```python id="gdho4j"
.readStream.table("Invoices_BZ")
```

Very simple.

---

# Silver Transformations

Silver layer still performs:

* explode invoices
* flatten structure

No changes in business logic.

---

# Silver Sink

Silver layer writes into:

```python id="u6u2g2"
Invoice_Line_Items
```

---

# Silver Query Name

```python id="kg44w9"
.queryName("silver_processing")
```

---

# Final Chained Streaming Architecture

Now we have:

```text id="zt7bjz"
Landing Zone
      ↓
Bronze Streaming Query
      ↓
Bronze Delta Table
      ↓
Silver Streaming Query
      ↓
Silver Delta Table
```

---

# Extremely Important Understanding

We now have:

# Two Independent Streaming Queries

running simultaneously.

---

# What Happens Internally?

---

## Bronze Query

Monitors landing zone.

---

## Silver Query

Monitors bronze Delta table.

---

# This Is Chained Streaming

One query feeds another query.

---

# Writing Test Suite

Now professor creates:

```python id="ly2ccj"
MedallionApproachTestSuite
```

---

# Test Environment Cleanup

Now cleanup becomes larger because we have:

* two tables
* two checkpoints
* archive directory
* landing zone

---

# Important Difference

Earlier we cleaned:

```text id="a5q0ow"
1 table
1 checkpoint
```

Now we clean:

```text id="zlx4ha"
2 tables
2 checkpoints
archive directories
```

because now we have two streaming jobs.

---

# Starting Streaming Queries

Professor starts:

## Bronze Query

and

## Silver Query

simultaneously.

---

# Important Insight

At startup:

* landing zone is empty
* bronze table may not exist

Still no problem.

---

# Why?

Because streaming queries are designed to:

# Wait for data

---

# Workflow During Test

---

## Step 1

Start Bronze query.

---

## Step 2

Start Silver query.

---

## Step 3

Ingest invoice file into landing zone.

---

# What Happens Next?

---

## Bronze Micro-Batch Starts

Reads invoice file.

Writes raw data into bronze table.

---

## Then Silver Micro-Batch Starts

Detects new bronze data.

Processes transformations.

Writes into silver table.

---

# Beautiful Chained Execution

```text id="1z69w6"
Landing Zone Updated
       ↓
Bronze Triggered
       ↓
Bronze Table Updated
       ↓
Silver Triggered
       ↓
Silver Table Updated
```

---

# Waiting for Micro-Batches

Professor waits for both queries to finish processing.

Then validates results.

---

# Multiple Iterations

Same process repeated for:

* first file
* second file
* third file

Each ingestion triggers:

* Bronze micro-batch
* Silver micro-batch

---

# Archive Validation

Professor also validates archival functionality.

---

# Important Observation

Suppose:

| File          | Processed By  |
| ------------- | ------------- |
| invoice1.json | Micro-batch 1 |
| invoice2.json | Micro-batch 2 |
| invoice3.json | Micro-batch 3 |

Then:

* first file archived by second micro-batch
* second file archived by third micro-batch
* third file NOT archived yet

because processing stops after third micro-batch.

---

# Important Archival Behavior

Archival happens during:

# Next Micro-Batch

not immediately after current one.

---

# Archive Directory Structure

Professor explains another subtle but important behavior.

Spark preserves:

# Original Directory Structure

inside archive location.

---

# Why?

Because multiple jobs may archive files into same archive base directory.

Preserving directory structure avoids collisions.

---

# Example

Suppose source path is:

```text id="s8bfmg"
/data/invoices/
```

Archive base:

```text id="c0e4rx"
/archive/
```

Spark internally creates:

```text id="swnt08"
/archive/data/invoices/
```

and stores archived files there.

---

# Spark UI Demonstration

Professor opens:

# Structured Streaming Tab

inside Spark UI.

---

# What Appears There?

Two running queries:

| Query Name        | Purpose         |
| ----------------- | --------------- |
| bronze_ingestion  | Raw ingestion   |
| silver_processing | Transformations |

---

# Why Query Names Matter?

Without names,
Spark UI becomes difficult to interpret.

With proper naming:

* debugging improves
* monitoring improves
* observability improves

---

# Metrics Visible in Spark UI

Spark UI shows:

* latest batch
* processing rate
* records processed
* query status
* execution metrics

---

# Final Shutdown

At end of tests:

```python id="dxj95w"
bronzeQuery.stop()
silverQuery.stop()
```

---

# Final Observation in Spark UI

Status becomes:

```text id="6e2mmb"
Finished
```

after stopping both queries.

---

# Most Important Learnings of This Lecture

# 1. Delta Tables Can Be Streaming Sources

Using:

```python id="kif00u"
readStream.table(...)
```

---

# 2. Streaming Jobs Can Be Chained

One streaming job can feed another.

---

# 3. Medallion Architecture Uses Chained Streams

Bronze → Silver → Gold

---

# 4. File Source Supports cleanSource

Allows:

* delete
* archive

processed files.

---

# 5. Query Naming Is Important

Helps monitor streaming jobs in Spark UI.

---

# 6. Multiple Streaming Queries Can Run Together

Spark supports many concurrent streaming jobs.

---

# Final Mental Model

Think about modern streaming architecture like this:

```text id="6js8a5"
Landing Zone
      ↓
Bronze Stream
      ↓
Bronze Delta Table
      ↓
Silver Stream
      ↓
Silver Delta Table
      ↓
Gold Stream
      ↓
Gold Analytics Table
```

This is the foundation of modern Lakehouse streaming systems.

See you again in the next lecture.

Keep learning and keep growing.
